In [ ]:
import sys
import subprocess
import pkgutil

def ensure_package(pkg_name, pip_name=None):
    if pkgutil.find_loader(pkg_name) is None:
        name = pip_name or pkg_name
        print(f"Installing {name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", name])
    else:
        print(f"{pkg_name} already installed.")

ensure_package("torch_geometric", "torch-geometric")


In [ ]:
from pathlib import Path

use_day_embedding = True
day_emb_dim       = 3

CONFIG = {
    "BASE_DIR"      : Path("."),
    "PATH_NETWORK"  : Path("./Elec Network Data.xlsx"),
    "PATH_WIND"     : Path("./input/Wind Data.xlsx"),
    "PATH_LOAD"     : Path("./input/Elec Load Data.xlsx"),
    "PATH_DECISION" : Path("./output/Decision_variablesRU.xlsx"),

    "MONTHS"   : [f"{m:02d}" for m in range(1, 13)],
    "NUM_NODES": 45,
    "GENERATOR_BUS_IDX": [6, 12, 14, 15, 18, 26, 27, 28, 32, 35, 37, 39, 41, 43, 44],
    "NUM_GENERATORS": 24,

    "HIST_HOURS"    : 72,
    "HORIZON_HOURS" : 24,
    "STRIDE_HOURS"  : 24,

    "TRAIN_END": "2024-10-01 00:00:00",
    "VAL_END"  : "2024-11-01 00:00:00",

    "USE_LOAD_HISTORY"         : True,
    "USE_WIND_OFFSHORE_HISTORY": True,
    "USE_WIND_ONSHORE_HISTORY" : True,
    "USE_P_HISTORY"            : False,

    "USE_NODE_STATIC_IS_GENERATOR"   : True,
    "USE_NODE_STATIC_GENERATOR_COUNT": True,
    "USE_NODE_STATIC_PMAX_TOTAL"     : True,
    "USE_NODE_STATIC_LOAD_SHARE"     : True,
    "USE_NODE_STATIC_PV_SHARE"       : True,
    "USE_NODE_STATIC_ONSHORE_SHARE"  : True,
    "USE_NODE_STATIC_IS_OFFSHORE"    : True,
    "USE_NODE_STATIC_IS_IMPORT"      : True,
    "USE_NODE_STATIC_IMPORT_PMAX"    : True,
    "USE_NODE_STATIC_VOLTAGE_KV"     : True,
    "USE_NODE_STATIC_IS_TRANSFER"    : True,
    "USE_NODE_STATIC_DEGREE"         : True,

    "EDGE_STATIC_COLUMNS": ["X_pu", "b_pu", "Pmax_MW", "Length_km", "Inom_A", "UB_kV"],
    "USE_BIDIRECTIONAL_EDGES": True,

    "MODEL_TYPE"          : "gine",
    "HIDDEN_DIM"          : 128,
    "ENCODER_HIDDEN_DIMS" : [256, 128],
    "DECODER_HIDDEN_DIMS" : [128, 64],
    "NUM_MP_LAYERS"       : 3,
    "DROPOUT"             : 0.15,
    "USE_BATCHNORM"       : True,
    "RESIDUAL"            : True,

    "SEED"                    : 42,
    "DEVICE"                  : "cpu",
    "EPOCHS"                  : 200,
    "BATCH_SIZE"              : 24,
    "LR"                      : 7e-4,
    "WEIGHT_DECAY"            : 1e-4,
    "GRAD_CLIP_NORM"          : 2.0,
    "EARLY_STOPPING_PATIENCE" : 30,
    "EARLY_STOPPING_MIN_DELTA": 1e-4,
    "SCHEDULER_FACTOR"        : 0.4,
    "SCHEDULER_PATIENCE"      : 6,
    "SCHEDULER_MIN_LR"        : 1e-6,

    "PERSISTENCE_MODE"     : "last_24h",
    "NUM_ALIGNMENT_CHECKS" : 5,
    "SAVE_MODEL_STATE"     : False,
    "MODEL_STATE_PATH"     : Path("spatial_elec_gnn.pt"),
}

PATHS = {
    "network" : CONFIG["BASE_DIR"] / CONFIG["PATH_NETWORK"],
    "wind"    : CONFIG["BASE_DIR"] / CONFIG["PATH_WIND"],
    "load"    : CONFIG["BASE_DIR"] / CONFIG["PATH_LOAD"],
    "decision": CONFIG["BASE_DIR"] / CONFIG["PATH_DECISION"],
}

TRAIN_END = CONFIG["TRAIN_END"]
VAL_END   = CONFIG["VAL_END"]
HIST      = CONFIG["HIST_HOURS"]
HOR       = CONFIG["HORIZON_HOURS"]
STRIDE    = CONFIG["STRIDE_HOURS"]
N_GEN     = len(CONFIG["GENERATOR_BUS_IDX"])

print("Fichiers attendus :")
for k, v in PATHS.items():
    print(f"  {k}: {v.resolve()} | exists={v.exists()}")
for p in PATHS.values():
    assert p.exists(), f"Fichier introuvable: {p}"
print("\nCONFIG OK.")
print(f"use_day_embedding={use_day_embedding}  day_emb_dim={day_emb_dim}")


In [ ]:
import copy
import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINEConv, NNConv

warnings.filterwarnings("ignore")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["SEED"])

DEVICE = torch.device(
    "cuda" if (CONFIG["DEVICE"] == "cuda" and torch.cuda.is_available()) else "cpu"
)
print("DEVICE =", DEVICE)

def pretty_shape(name, x):
    if isinstance(x, (np.ndarray, torch.Tensor, pd.DataFrame, pd.Series)):
        print(f"{name}: shape={getattr(x, 'shape', None)}")
    else:
        print(f"{name}: type={type(x)}")

def drop_unnamed_first_col(df):
    if len(df.columns) > 0 and str(df.columns[0]).startswith("Unnamed"):
        return df.drop(columns=[df.columns[0]])
    return df

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


In [ ]:
buses_df   = pd.read_excel(PATHS["network"], sheet_name="Buses")
lines_df   = pd.read_excel(PATHS["network"], sheet_name="LinesMerged")
gen_df     = pd.read_excel(PATHS["network"], sheet_name="Gen")
genmap_df  = pd.read_excel(PATHS["network"], sheet_name="GenMap",  header=0)
windmap_df = pd.read_excel(PATHS["network"], sheet_name="WindMap", header=0)
impmap_df  = pd.read_excel(PATHS["network"], sheet_name="ImpMap",  header=0)
imp_df     = pd.read_excel(PATHS["network"], sheet_name="Imp")
renew_df   = pd.read_excel(PATHS["network"], sheet_name="Renew")

num_nodes         = CONFIG["NUM_NODES"]
generator_bus_idx = CONFIG["GENERATOR_BUS_IDX"]

for name, df in [("buses_df", buses_df), ("lines_df", lines_df),
                 ("gen_df", gen_df), ("genmap_df", genmap_df),
                 ("windmap_df", windmap_df), ("impmap_df", impmap_df)]:
    pretty_shape(name, df)

assert len(buses_df)  == num_nodes
assert len(gen_df)    == CONFIG["NUM_GENERATORS"]
assert genmap_df.shape == (num_nodes, CONFIG["NUM_GENERATORS"])
assert windmap_df.shape[0] == num_nodes

print("\nColonnes Buses :", list(buses_df.columns))
print("Colonnes Lines :", list(lines_df.columns))
print("Colonnes Gen   :", list(gen_df.columns))
display(buses_df.head())
display(lines_df.head())
display(gen_df)


In [ ]:
genmap = genmap_df.to_numpy(dtype=float)

generator_to_bus = {}
for gen_idx in range(CONFIG["NUM_GENERATORS"]):
    connected = np.where(genmap[:, gen_idx] > 0.5)[0]
    assert len(connected) == 1, f"Gen {gen_idx} connecté à {len(connected)} bus"
    generator_to_bus[gen_idx] = int(connected[0])

print("Mapping gen → bus (0-based):")
for g, b in generator_to_bus.items():
    print(f"  Gen{g+1:2d} → Bus {b+1:2d} ({buses_df.iloc[b]['Name']})")

pmax_per_bus = np.zeros(num_nodes, dtype=float)
for gen_idx, bus_idx in generator_to_bus.items():
    pmax_val = gen_df.iloc[gen_idx]["Pmax_MW"]
    if isinstance(pmax_val, str):
        pmax_val = eval(pmax_val.replace("=", ""))
    pmax_per_bus[bus_idx] += float(pmax_val)

gen_count_per_bus = np.zeros(num_nodes, dtype=int)
for gen_idx, bus_idx in generator_to_bus.items():
    gen_count_per_bus[bus_idx] += 1

gen_buses_from_map = sorted(set(generator_to_bus.values()))
assert gen_buses_from_map == sorted(CONFIG["GENERATOR_BUS_IDX"])
print(f"\n{len(gen_buses_from_map)} bus générateurs (0-based): {gen_buses_from_map}")

windmap        = windmap_df.to_numpy(dtype=float).flatten()
offshore_bus_idx = np.where(windmap > 0.5)[0]
assert len(offshore_bus_idx) == 1 and offshore_bus_idx[0] == 38
print(f"Bus offshore : bus {offshore_bus_idx[0]} (STEVN), windmap OK")

impmap = impmap_df.to_numpy(dtype=float)
is_import_bus  = (impmap.sum(axis=1) > 0).astype(float)
import_pmax_mw = np.zeros(num_nodes, dtype=float)
for i in range(num_nodes):
    for j in range(8):
        if impmap[i, j] > 0.5:
            import_pmax_mw[i] += float(imp_df.iloc[j]["Pmax_MW"])

print(f"Bus d'interconnexion (1-based): {np.where(is_import_bus > 0)[0] + 1}")


In [ ]:
transfos   = buses_df["Transfos"].to_numpy(dtype=float)
load_share = transfos / transfos.sum()
assert np.isclose(load_share.sum(), 1.0)
print("load_share.sum() =", load_share.sum())

PV_SHARES_RAW = {
    1: 3.455/2,  2: 2.192/4,  4: 16.02/2,  5: 7.093/11,
    7: 15.46/3,  8: 7.093/11, 10: 3.455/2, 13: 17.37/4,
   14: 17.37/4, 16: 7.625/2, 17: 7.093/11, 19: 16.02/2,
   20: 7.093/11, 21: 17.26/4, 22: 7.093/11, 23: 7.093/11,
   25: 2.192/4, 26: 17.26/4, 27: 17.26/4,  28: 15.46/3,
   29: 17.37/4, 31: 7.093/11, 32: 7.093/11, 33: 17.37/4,
   34: 7.093/11, 35: 2.192/4, 36: 7.093/11, 38: 7.625/2,
   40: 7.093/11, 42: 15.46/3, 43: 2.192/4, 44: 17.26/4,
   45: 13.52,
}
ONSHORE_SHARES_RAW = {
    1: 11.599/2,  2: 3.459/4,  4: 8.741/2,  5: 9.825/11,
    7: 5.15/3,   8: 9.825/11, 10: 11.599/2, 13: 18.252/4,
   14: 18.252/4, 16: 17.785/2, 17: 9.825/11, 19: 8.741/2,
   20: 9.825/11, 21: 14.835/4, 22: 9.825/11, 23: 9.825/11,
   25: 3.459/4,  26: 14.835/4, 27: 14.835/4, 28: 5.15/3,
   29: 18.252/4, 31: 9.825/11, 32: 9.825/11, 33: 18.252/4,
   34: 9.825/11, 35: 3.459/4,  36: 9.825/11, 38: 17.785/2,
   40: 9.825/11, 42: 5.15/3,  43: 3.459/4,  44: 14.835/4,
   45: 10.35,
}

pv_share      = np.array([PV_SHARES_RAW.get(i+1, 0.0)      for i in range(num_nodes)], dtype=float)
onshore_share = np.array([ONSHORE_SHARES_RAW.get(i+1, 0.0) for i in range(num_nodes)], dtype=float)

pv_share_norm      = pv_share / (pv_share.sum() + 1e-12)
onshore_share_norm = onshore_share / (onshore_share.sum() + 1e-12)

print("pv_share.sum()      =", pv_share.sum())
print("onshore_share.sum() =", onshore_share.sum())

display(pd.DataFrame({
    "bus_1based"   : np.arange(1, 46),
    "name"         : buses_df["Name"].to_numpy(),
    "type"         : buses_df["Type"].to_numpy(),
    "load_share"   : load_share,
    "pv_share"     : pv_share_norm,
    "onshore_share": onshore_share_norm,
    "gen_count"    : gen_count_per_bus,
    "pmax_total"   : pmax_per_bus,
    "is_import"    : is_import_bus.astype(int),
    "is_offshore"  : windmap.astype(int),
}).head(20))


In [ ]:
def read_decision_p(months, decision_path):
    all_arrays = []
    for m in months:
        df = pd.read_excel(decision_path, sheet_name=f"p-{m}-2024", header=0, index_col=0)
        all_arrays.append(df.to_numpy(dtype=float))
    p_full = np.concatenate(all_arrays, axis=0)
    ts_p   = pd.date_range("2024-01-01", periods=len(p_full), freq="h")
    return p_full, ts_p

def read_elec_load(months, load_path):
    dfs = []
    for m in months:
        df = pd.read_excel(load_path, sheet_name=f"{m}-2024", header=0, usecols=[0, 1])
        df.columns = ["datetime", "load_MW"]
        df["datetime"] = pd.to_datetime(df["datetime"])
        dfs.append(df[["datetime", "load_MW"]])
    df_all = pd.concat(dfs, ignore_index=True).sort_values("datetime").reset_index(drop=True)
    return df_all["load_MW"].to_numpy(dtype=float), pd.DatetimeIndex(df_all["datetime"])

def read_wind(months, wind_path):
    dfs = []
    for m in months:
        df = pd.read_excel(wind_path, sheet_name=f"{m}-2024", header=0, usecols=[0, 1])
        df.columns = ["datetime", "wind_MW"]
        df["datetime"] = pd.to_datetime(df["datetime"])
        dfs.append(df[["datetime", "wind_MW"]])
    df_all = pd.concat(dfs, ignore_index=True).sort_values("datetime").reset_index(drop=True)
    return df_all["wind_MW"].to_numpy(dtype=float), pd.DatetimeIndex(df_all["datetime"])


In [ ]:
p_full,     ts_p    = read_decision_p(CONFIG["MONTHS"], PATHS["decision"])
load_total, ts_load = read_elec_load(CONFIG["MONTHS"],  PATHS["load"])
wind_total, ts_wind = read_wind(CONFIG["MONTHS"],       PATHS["wind"])

assert len(ts_p) == len(ts_load) == len(ts_wind)
assert p_full.shape[1] == 24

T = len(ts_p)
print("Plage temporelle :", ts_p.min(), "->", ts_p.max())
print("Nombre total d'heures :", T)

pretty_shape("p_full [T, 24]",   p_full)
pretty_shape("load_total [T]",   load_total)
pretty_shape("wind_total [T]",   wind_total)


In [ ]:
p_bus_full = np.zeros((T, num_nodes), dtype=float)
for gen_idx, bus_idx in generator_to_bus.items():
    p_bus_full[:, bus_idx] += p_full[:, gen_idx]

p_bus_generator = p_bus_full[:, CONFIG["GENERATOR_BUS_IDX"]]

np.testing.assert_allclose(p_bus_full.sum(axis=1), p_full.sum(axis=1), rtol=1e-6)
print("Agrégation p par bus OK.")

load_nodal          = load_total[:, None] * load_share[None, :]
wind_offshore_nodal = np.zeros((T, num_nodes), dtype=float)
wind_offshore_nodal[:, 38] = wind_total
wind_onshore_nodal  = wind_total[:, None] * onshore_share_norm[None, :]
p_history_nodal     = p_bus_full.copy()

pretty_shape("p_bus_generator [T, 15]",    p_bus_generator)
pretty_shape("load_nodal [T, 45]",         load_nodal)
pretty_shape("wind_offshore_nodal [T, 45]",wind_offshore_nodal)


In [ ]:
dyn_feature_dict = {}

if CONFIG["USE_LOAD_HISTORY"]:
    dyn_feature_dict["load_nodal"] = load_nodal
if CONFIG["USE_WIND_OFFSHORE_HISTORY"]:
    dyn_feature_dict["wind_offshore"] = wind_offshore_nodal
if CONFIG["USE_WIND_ONSHORE_HISTORY"]:
    dyn_feature_dict["wind_onshore"] = wind_onshore_nodal
if CONFIG["USE_P_HISTORY"]:
    dyn_feature_dict["p_history"] = p_history_nodal

assert len(dyn_feature_dict) > 0, "Au moins une feature dynamique doit être activée."

dyn_feature_names = list(dyn_feature_dict.keys())
dyn_tensor        = np.stack([dyn_feature_dict[k] for k in dyn_feature_names], axis=-1)

print("Features dynamiques :", dyn_feature_names)
pretty_shape("dyn_tensor [T, N, F_dyn]", dyn_tensor)

display(pd.DataFrame({
    "feature": dyn_feature_names,
    "shape"  : [str(dyn_feature_dict[k].shape) for k in dyn_feature_names],
    "min"    : [dyn_feature_dict[k].min()  for k in dyn_feature_names],
    "max"    : [dyn_feature_dict[k].max()  for k in dyn_feature_names],
    "mean"   : [dyn_feature_dict[k].mean() for k in dyn_feature_names],
}))


In [ ]:
degrees = np.zeros(num_nodes, dtype=float)
for _, row in lines_df.iterrows():
    u = int(row["FromNode"]) - 1
    v = int(row["ToNode"])   - 1
    degrees[u] += 1
    degrees[v] += 1

is_transfer = (buses_df["Type"] == "Transfer").to_numpy(dtype=float)
voltage_kv  = buses_df["UB_kV"].to_numpy(dtype=float)

node_static_parts = []
node_static_names = []

_ns = [
    ("USE_NODE_STATIC_IS_GENERATOR",    lambda: (gen_count_per_bus > 0).astype(float)[:, None],          "is_generator"),
    ("USE_NODE_STATIC_GENERATOR_COUNT", lambda: gen_count_per_bus[:, None].astype(float),                "generator_count"),
    ("USE_NODE_STATIC_PMAX_TOTAL",      lambda: pmax_per_bus[:, None],                                   "pmax_total_MW"),
    ("USE_NODE_STATIC_LOAD_SHARE",      lambda: load_share[:, None],                                     "load_share"),
    ("USE_NODE_STATIC_PV_SHARE",        lambda: pv_share_norm[:, None],                                  "pv_share"),
    ("USE_NODE_STATIC_ONSHORE_SHARE",   lambda: onshore_share_norm[:, None],                             "onshore_share"),
    ("USE_NODE_STATIC_IS_OFFSHORE",     lambda: windmap[:, None],                                        "is_offshore_wind"),
    ("USE_NODE_STATIC_IS_IMPORT",       lambda: is_import_bus[:, None],                                  "is_import_bus"),
    ("USE_NODE_STATIC_IMPORT_PMAX",     lambda: import_pmax_mw[:, None],                                 "import_pmax_MW"),
    ("USE_NODE_STATIC_VOLTAGE_KV",      lambda: voltage_kv[:, None],                                     "voltage_kV"),
    ("USE_NODE_STATIC_IS_TRANSFER",     lambda: is_transfer[:, None],                                    "is_transfer"),
    ("USE_NODE_STATIC_DEGREE",          lambda: degrees[:, None],                                        "degree"),
]

for key, builder, name in _ns:
    if CONFIG.get(key, False):
        node_static_parts.append(builder())
        node_static_names.append(name)

node_static = (
    np.concatenate(node_static_parts, axis=1)
    if node_static_parts
    else np.zeros((num_nodes, 0), dtype=float)
)

pretty_shape("node_static [N, F_static]", node_static)
print("node_static_names =", node_static_names)


In [ ]:
edge_static_cols = CONFIG["EDGE_STATIC_COLUMNS"]
edge_static_base = (
    lines_df[edge_static_cols]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0.0)
    .to_numpy(dtype=float)
)

edge_pairs = []
edge_attrs = []
for i, row in lines_df.iterrows():
    u  = int(row["FromNode"]) - 1
    v  = int(row["ToNode"])   - 1
    ea = edge_static_base[i]
    edge_pairs.append((u, v))
    edge_attrs.append(ea)
    if CONFIG["USE_BIDIRECTIONAL_EDGES"]:
        edge_pairs.append((v, u))
        edge_attrs.append(ea.copy())

edge_index    = torch.tensor(np.array(edge_pairs, dtype=np.int64).T, dtype=torch.long)
edge_attr_raw = np.array(edge_attrs, dtype=float)

assert edge_index.shape[0] == 2
assert edge_index.max() < num_nodes
assert edge_attr_raw.shape == (edge_index.shape[1], len(edge_static_cols))

print("edge_static_cols =", edge_static_cols)
print("Nb arêtes :", edge_index.shape[1])
pretty_shape("edge_index", edge_index)
pretty_shape("edge_attr_raw", edge_attr_raw)


In [ ]:
ts_p = pd.to_datetime(ts_p)

day_of_week = ts_p.dayofweek.to_numpy(dtype=np.int64)
is_weekend  = (day_of_week >= 5).astype(np.int64)

print("Plage temporelle       :", ts_p.min(), "->", ts_p.max())
print("day_of_week [0=Mon … 6=Sun]  min/max:", day_of_week.min(), "/", day_of_week.max())
print("is_weekend (0/1)             sum:", is_weekend.sum(), "heures week-end")

pd.DataFrame({
    "datetime"   : ts_p[:8],
    "day_of_week": day_of_week[:8],
    "is_weekend" : is_weekend[:8],
}).pipe(display)


In [ ]:
TRAIN_END_ts = pd.Timestamp(TRAIN_END)
VAL_END_ts   = pd.Timestamp(VAL_END)

train_time_mask = ts_p < TRAIN_END_ts
val_time_mask   = (ts_p >= TRAIN_END_ts) & (ts_p < VAL_END_ts)
test_time_mask  = ts_p >= VAL_END_ts

print("Train hours :", int(train_time_mask.sum()))
print("Val hours   :", int(val_time_mask.sum()))
print("Test hours  :", int(test_time_mask.sum()))

T, N_nodes, F_dyn = dyn_tensor.shape

dyn_scaler = StandardScaler()
dyn_scaler.fit(dyn_tensor[train_time_mask].reshape(-1, F_dyn))
dyn_tensor_scaled = dyn_scaler.transform(dyn_tensor.reshape(-1, F_dyn)).reshape(T, N_nodes, F_dyn)

node_static_scaler = StandardScaler()
node_static_scaler.fit(node_static)
node_static_scaled = node_static_scaler.transform(node_static)

edge_scaler = StandardScaler()
edge_scaler.fit(edge_attr_raw)
edge_attr_scaled = edge_scaler.transform(edge_attr_raw)

target_scaler = StandardScaler()
target_scaler.fit(p_bus_generator[train_time_mask])
p_bus_gen_scaled = target_scaler.transform(p_bus_generator)

print("\nTarget scaler mean shape :", target_scaler.mean_.shape)
print("Target scaler scale shape:", target_scaler.scale_.shape)
pretty_shape("dyn_tensor_scaled    [T, N_nodes, F_dyn]", dyn_tensor_scaled)
pretty_shape("node_static_scaled   [N_nodes, F_static]", node_static_scaled)
pretty_shape("edge_attr_scaled     [N_edges, F_edge]",   edge_attr_scaled)
pretty_shape("p_bus_gen_scaled     [T, N_GEN]",          p_bus_gen_scaled)
print("\nNormalisation OK.")


In [ ]:
all_start_indices = np.arange(0, T - HIST - HOR + 1, STRIDE)

forecast_starts   = pd.DatetimeIndex([ts_p[i + HIST] for i in all_start_indices])
train_window_mask = forecast_starts < TRAIN_END_ts
val_window_mask   = (forecast_starts >= TRAIN_END_ts) & (forecast_starts < VAL_END_ts)
test_window_mask  = forecast_starts >= VAL_END_ts

print(f"Nombre total de fenêtres : {len(all_start_indices)}")
print(f"  Train : {int(train_window_mask.sum())}")
print(f"  Val   : {int(val_window_mask.sum())}")
print(f"  Test  : {int(test_window_mask.sum())}")

expected_x_dim = HIST * F_dyn + node_static_scaled.shape[1]
print(f"\nexpected_x_dim (sans embedding) = {expected_x_dim}")
print(f"N_GEN = {N_GEN}")

def build_single_window(start_idx):
    hist_dyn = dyn_tensor_scaled[start_idx : start_idx + HIST]
    hist_dyn = np.transpose(hist_dyn, (1, 0, 2)).reshape(num_nodes, -1)
    x        = np.concatenate([hist_dyn, node_static_scaled], axis=1)

    y      = p_bus_gen_scaled[start_idx + HIST : start_idx + HIST + HOR].T
    y_real = p_bus_generator[start_idx + HIST  : start_idx + HIST + HOR].T

    dow_idx = int(day_of_week[start_idx + HIST])

    return {
        "x"                : x.astype(np.float32),
        "y"                : y.astype(np.float32),
        "y_real"           : y_real.astype(np.float32),
        "dow"              : dow_idx,
        "start_idx"        : np.int64(start_idx),
        "target_start_idx" : np.int64(start_idx + HIST),
        "target_end_idx"   : np.int64(start_idx + HIST + HOR - 1),
    }

sample0 = build_single_window(int(all_start_indices[0]))
assert sample0["x"].shape == (num_nodes, expected_x_dim)
assert sample0["y"].shape == (N_GEN, HOR)
print("\nbuild_single_window OK.")

def check_alignment(start_idx, verbose=True):
    s = build_single_window(start_idx)
    ts0, ts1 = int(s["target_start_idx"]), int(s["target_end_idx"])
    expected = p_bus_generator[ts0:ts1+1].T
    assert np.allclose(expected, s["y_real"]), "Problème d'alignement y_real"
    if verbose:
        print(f"Alignment OK | start={start_idx} | {ts_p[ts0]} → {ts_p[ts1]}")

for idx in [all_start_indices[0], all_start_indices[10], all_start_indices[-1]]:
    check_alignment(int(idx))


In [ ]:
class ElecSpatialWindowDataset(torch.utils.data.Dataset):
    def __init__(self, start_indices):
        self.start_indices = np.array(start_indices, dtype=int)

    def __len__(self):
        return len(self.start_indices)

    def __getitem__(self, idx):
        s = build_single_window(int(self.start_indices[idx]))
        data = Data(
            x          = torch.tensor(s["x"],           dtype=torch.float32),
            edge_index = edge_index.clone(),
            edge_attr  = torch.tensor(edge_attr_scaled, dtype=torch.float32),
            y          = torch.tensor(s["y"],           dtype=torch.float32),
            generator_mask = torch.tensor(
                np.isin(np.arange(num_nodes), CONFIG["GENERATOR_BUS_IDX"]),
                dtype=torch.bool
            ),
        )
        data.dow              = int(s["dow"])
        data.start_idx        = int(s["start_idx"])
        data.target_start_idx = int(s["target_start_idx"])
        data.target_end_idx   = int(s["target_end_idx"])
        data.y_real           = torch.tensor(s["y_real"], dtype=torch.float32)
        return data

train_ds = ElecSpatialWindowDataset(all_start_indices[train_window_mask])
val_ds   = ElecSpatialWindowDataset(all_start_indices[val_window_mask])
test_ds  = ElecSpatialWindowDataset(all_start_indices[test_window_mask])

train_loader = DataLoader(train_ds, batch_size=CONFIG["BATCH_SIZE"], shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG["BATCH_SIZE"], shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=CONFIG["BATCH_SIZE"], shuffle=False)

print(f"len(train_ds) = {len(train_ds)}")
print(f"len(val_ds)   = {len(val_ds)}")
print(f"len(test_ds)  = {len(test_ds)}")

batch = next(iter(train_loader))
print(batch)
pretty_shape("batch.x",              batch.x)
pretty_shape("batch.edge_attr",      batch.edge_attr)
pretty_shape("batch.y",              batch.y)
pretty_shape("batch.generator_mask", batch.generator_mask)
assert batch.x.shape[1]         == expected_x_dim
assert batch.edge_attr.shape[1] == edge_attr_scaled.shape[1]
assert batch.y.shape[-1]        == HOR


In [ ]:
class MLP(nn.Module):
    def __init__(self, dims, dropout=0.0, use_batchnorm=False, activate_last=False):
        super().__init__()
        self.dims         = list(dims)
        self.in_channels  = self.dims[0]
        self.out_channels = self.dims[-1]
        layers = []
        for i in range(len(dims) - 1):
            layers.append(nn.Linear(dims[i], dims[i+1]))
            is_last = (i == len(dims) - 2)
            if (not is_last) or activate_last:
                if use_batchnorm:
                    layers.append(nn.BatchNorm1d(dims[i+1]))
                layers.append(nn.ReLU())
                if dropout > 0:
                    layers.append(nn.Dropout(dropout))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


In [ ]:
class SpatialElecGNN(nn.Module):
    def __init__(
        self,
        in_dim,
        edge_dim,
        hidden_dim,
        horizon,
        model_type="gine",
        encoder_hidden_dims=(256, 128),
        decoder_hidden_dims=(128, 64),
        num_mp_layers=3,
        dropout=0.0,
        use_batchnorm=True,
        residual=True,
        use_day_embedding=False,
        day_emb_dim=3,
    ):
        super().__init__()
        self.model_type       = model_type.lower()
        self.residual         = residual
        self.num_mp_layers    = num_mp_layers
        self.use_day_embedding = use_day_embedding

        if use_day_embedding:
            self.day_embedding = nn.Embedding(7, day_emb_dim)
            enc_in_dim = in_dim + day_emb_dim
        else:
            enc_in_dim = in_dim

        enc_dims = [enc_in_dim] + list(encoder_hidden_dims) + [hidden_dim]
        self.encoder = MLP(enc_dims, dropout=dropout,
                           use_batchnorm=use_batchnorm, activate_last=False)

        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_mp_layers):
            if self.model_type == "nnconv":
                edge_net = MLP([edge_dim, hidden_dim, hidden_dim * hidden_dim],
                               dropout=dropout, use_batchnorm=False, activate_last=False)
                conv = NNConv(hidden_dim, hidden_dim, nn=edge_net, aggr="mean")
            elif self.model_type == "gine":
                nn_mlp = MLP([hidden_dim, hidden_dim, hidden_dim],
                              dropout=dropout, use_batchnorm=False, activate_last=False)
                nn_mlp.in_channels = hidden_dim
                conv = GINEConv(nn=nn_mlp, edge_dim=edge_dim)
            else:
                raise ValueError(f"MODEL_TYPE inconnu: {model_type}")
            self.convs.append(conv)
            self.norms.append(
                nn.BatchNorm1d(hidden_dim) if use_batchnorm else nn.Identity()
            )

        dec_dims = [hidden_dim] + list(decoder_hidden_dims) + [horizon]
        self.decoder = MLP(dec_dims, dropout=dropout,
                           use_batchnorm=use_batchnorm, activate_last=False)

    def forward(self, x, edge_index, edge_attr, generator_mask, dow=None, batch_idx=None):
        if self.use_day_embedding and dow is not None and batch_idx is not None:
            dow_emb      = self.day_embedding(dow)
            dow_expanded = dow_emb[batch_idx]
            x = torch.cat([x, dow_expanded], dim=1)

        h = self.encoder(x)
        for conv, norm in zip(self.convs, self.norms):
            h_in = h
            h    = conv(h, edge_index, edge_attr)
            h    = norm(h)
            h    = F.relu(h)
            if self.residual:
                h = h + h_in

        h_gen = h[generator_mask]
        return self.decoder(h_gen)


model = SpatialElecGNN(
    in_dim              = expected_x_dim,
    edge_dim            = edge_attr_scaled.shape[1],
    hidden_dim          = CONFIG["HIDDEN_DIM"],
    horizon             = HOR,
    model_type          = CONFIG["MODEL_TYPE"],
    encoder_hidden_dims = CONFIG["ENCODER_HIDDEN_DIMS"],
    decoder_hidden_dims = CONFIG["DECODER_HIDDEN_DIMS"],
    num_mp_layers       = CONFIG["NUM_MP_LAYERS"],
    dropout             = CONFIG["DROPOUT"],
    use_batchnorm       = CONFIG["USE_BATCHNORM"],
    residual            = CONFIG["RESIDUAL"],
    use_day_embedding   = use_day_embedding,
    day_emb_dim         = day_emb_dim,
).to(DEVICE)

print(model)

test_batch = next(iter(train_loader)).to(DEVICE)
with torch.no_grad():
    dow_t   = torch.tensor(test_batch.dow, dtype=torch.long).to(DEVICE) if isinstance(test_batch.dow, list) else test_batch.dow.to(DEVICE)
    test_pred = model(
        test_batch.x, test_batch.edge_index, test_batch.edge_attr,
        test_batch.generator_mask,
        dow=dow_t if use_day_embedding else None,
        batch_idx=test_batch.batch if use_day_embedding else None,
    )
print("\nForward OK  test_pred.shape =", test_pred.shape)
assert test_pred.shape == test_batch.y.view(-1, HOR).shape


In [ ]:
def mse_loss_on_generators(pred, batch):
    y_true = batch.y.view_as(pred)
    return F.mse_loss(pred, y_true)

def inverse_targets_flexible(arr_scaled, n_gen=N_GEN, horizon=HOR):
    arr = np.asarray(arr_scaled)
    scaler_shape = np.asarray(target_scaler.mean_).shape

    if scaler_shape in [(1,), ()]:
        return target_scaler.inverse_transform(arr.reshape(-1, 1)).reshape(arr.shape)

    if len(scaler_shape) == 1 and scaler_shape[0] == n_gen:
        if arr.ndim == 2:
            bsz  = arr.shape[0] // n_gen
            arr2 = arr.reshape(bsz, n_gen, horizon).transpose(0, 2, 1).reshape(-1, n_gen)
            return target_scaler.inverse_transform(arr2).reshape(bsz, horizon, n_gen).transpose(0, 2, 1).reshape(arr.shape)
        if arr.ndim == 3:
            bsz  = arr.shape[0]
            arr2 = arr.transpose(0, 2, 1).reshape(-1, n_gen)
            return target_scaler.inverse_transform(arr2).reshape(bsz, horizon, n_gen).transpose(0, 2, 1)

    raise ValueError(f"target_scaler incompatible: mean_.shape={scaler_shape}")

def mae_rmse_nmae(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true).reshape(-1), np.asarray(y_pred).reshape(-1)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    nmae = mae / (np.mean(np.abs(y_true)) + 1e-12)
    return {"MAE": mae, "RMSE": rmse, "NMAE": nmae}

def persistence_forecast_for_sample(start_idx, mode=None):
    mode   = mode or CONFIG["PERSISTENCE_MODE"]
    hist_p = p_bus_generator[start_idx : start_idx + HIST]
    if mode == "last_24h":
        return hist_p[-HOR:].T
    if mode == "last_value":
        return np.repeat(hist_p[[-1]], HOR, axis=0).T
    raise ValueError(f"PERSISTENCE_MODE inconnu: {mode}")

def run_one_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, total_samples = 0.0, 0
    for batch in loader:
        batch = batch.to(DEVICE)
        dow_t = batch.dow.to(DEVICE) if hasattr(batch.dow, "to") else torch.tensor(batch.dow, dtype=torch.long).to(DEVICE)
        with torch.set_grad_enabled(is_train):
            pred = model(
                batch.x, batch.edge_index, batch.edge_attr, batch.generator_mask,
                dow=dow_t if use_day_embedding else None,
                batch_idx=batch.batch if use_day_embedding else None,
            )
            loss = mse_loss_on_generators(pred, batch)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                if CONFIG["GRAD_CLIP_NORM"] is not None:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["GRAD_CLIP_NORM"])
                optimizer.step()
        total_loss    += loss.item() * batch.num_graphs
        total_samples += batch.num_graphs
    return total_loss / max(total_samples, 1)


In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.0):
        self.patience    = patience
        self.min_delta   = min_delta
        self.best_loss   = np.inf
        self.best_state  = None
        self.counter     = 0
        self.should_stop = False

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss  = float(val_loss)
            self.best_state = copy.deepcopy(model.state_dict())
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        return self.should_stop


In [ ]:
model = SpatialElecGNN(
    in_dim              = expected_x_dim,
    edge_dim            = edge_attr_scaled.shape[1],
    hidden_dim          = CONFIG["HIDDEN_DIM"],
    horizon             = HOR,
    model_type          = CONFIG["MODEL_TYPE"],
    encoder_hidden_dims = CONFIG["ENCODER_HIDDEN_DIMS"],
    decoder_hidden_dims = CONFIG["DECODER_HIDDEN_DIMS"],
    num_mp_layers       = CONFIG["NUM_MP_LAYERS"],
    dropout             = CONFIG["DROPOUT"],
    use_batchnorm       = CONFIG["USE_BATCHNORM"],
    residual            = CONFIG["RESIDUAL"],
    use_day_embedding   = use_day_embedding,
    day_emb_dim         = day_emb_dim,
).to(DEVICE)

optimizer = torch.optim.Adam(
    model.parameters(), lr=CONFIG["LR"], weight_decay=CONFIG["WEIGHT_DECAY"]
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=CONFIG["SCHEDULER_FACTOR"],
    patience=CONFIG["SCHEDULER_PATIENCE"], min_lr=CONFIG["SCHEDULER_MIN_LR"],
)
early_stopper = EarlyStopping(
    patience  = CONFIG["EARLY_STOPPING_PATIENCE"],
    min_delta = CONFIG["EARLY_STOPPING_MIN_DELTA"],
)
history = {"train_loss": [], "val_loss": [], "lr": []}

print(model)
print("\nModèle / optimizer / scheduler / history réinitialisés.")


In [ ]:
for epoch in range(1, CONFIG["EPOCHS"] + 1):
    train_loss = run_one_epoch(model, train_loader, optimizer=optimizer)
    val_loss   = run_one_epoch(model, val_loader,   optimizer=None)

    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["lr"].append(current_lr)

    print(
        f"Epoch {epoch:03d} | "
        f"train={train_loss:.6f} | val={val_loss:.6f} | "
        f"counter={early_stopper.counter}/{early_stopper.patience} | lr={current_lr:.2e}"
    )

    if early_stopper.step(val_loss, model):
        print(f"Early stopping déclenché à l'epoch {epoch}.")
        break

assert early_stopper.best_state is not None
model.load_state_dict(early_stopper.best_state)
print("\nMeilleur val_loss:", early_stopper.best_loss)

if CONFIG["SAVE_MODEL_STATE"]:
    torch.save(model.state_dict(), CONFIG["MODEL_STATE_PATH"])
    print("Modèle sauvegardé dans", CONFIG["MODEL_STATE_PATH"])


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
epochs_ran = range(1, len(history["train_loss"]) + 1)
ax1.plot(epochs_ran, history["train_loss"], label="train")
ax1.plot(epochs_ran, history["val_loss"],   label="val")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("MSE Loss")
ax1.set_title("Courbes de perte"); ax1.legend(); ax1.grid(True)

ax2.plot(epochs_ran, history["lr"])
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Learning rate")
ax2.set_title("Learning rate schedule"); ax2.grid(True)

plt.tight_layout()
plt.show()
print(f"Entraînement terminé en {len(history['train_loss'])} epochs.")
print(f"Best val_loss = {early_stopper.best_loss:.6f}")


In [ ]:
def ensure_3d_output(arr, n_gen, horizon):
    arr = np.asarray(arr)
    if arr.ndim == 3:
        assert arr.shape[1] == n_gen and arr.shape[2] == horizon
        return arr
    if arr.ndim == 2:
        assert arr.shape[1] == horizon
        assert arr.shape[0] % n_gen == 0
        return arr.reshape(arr.shape[0] // n_gen, n_gen, horizon)
    raise ValueError(f"ndim non supporté: {arr.ndim}")

@torch.no_grad()
def collect_predictions(mdl, loader):
    mdl.eval()
    preds_scaled, trues_scaled, preds_real, trues_real, metas = [], [], [], [], []
    for batch in loader:
        batch = batch.to(DEVICE)
        dow_t = batch.dow.to(DEVICE) if hasattr(batch.dow, "to") else torch.tensor(batch.dow, dtype=torch.long).to(DEVICE)
        pred  = mdl(
            batch.x, batch.edge_index, batch.edge_attr, batch.generator_mask,
            dow=dow_t if use_day_embedding else None,
            batch_idx=batch.batch if use_day_embedding else None,
        )
        pred_np = pred.detach().cpu().numpy()
        true_np = batch.y.detach().cpu().numpy()
        preds_scaled.append(pred_np)
        trues_scaled.append(true_np)
        preds_real.append(inverse_targets_flexible(pred_np))
        trues_real.append(inverse_targets_flexible(true_np))
        if hasattr(batch, "target_start_idx"):
            metas.append(batch.target_start_idx.detach().cpu().numpy())

    return {
        "preds_scaled": np.concatenate(preds_scaled, axis=0),
        "trues_scaled": np.concatenate(trues_scaled, axis=0),
        "preds_real"  : np.concatenate(preds_real,   axis=0),
        "trues_real"  : np.concatenate(trues_real,   axis=0),
        "metas"       : np.concatenate(metas, axis=0) if metas else np.array([]),
    }

train_out = collect_predictions(model, train_loader)
val_out   = collect_predictions(model, val_loader)
test_out  = collect_predictions(model, test_loader)

for split, out in [("train", train_out), ("val", val_out), ("test", test_out)]:
    print(f"{split} | preds_real={out['preds_real'].shape} | trues_real={out['trues_real'].shape}")


In [ ]:
def compute_persistence_split(dataset, mode=None):
    mode = mode or CONFIG["PERSISTENCE_MODE"]
    preds, trues = [], []
    for start_idx in dataset.start_indices:
        start_idx    = int(start_idx)
        target_start = start_idx + HIST
        pred = persistence_forecast_for_sample(start_idx, mode=mode)
        true = p_bus_generator[target_start : target_start + HOR].T
        preds.append(pred)
        trues.append(true)
    return np.stack(preds, axis=0), np.stack(trues, axis=0)

pers_train_pred, pers_train_true = compute_persistence_split(train_ds)
pers_val_pred,   pers_val_true   = compute_persistence_split(val_ds)
pers_test_pred,  pers_test_true  = compute_persistence_split(test_ds)

print("pers_train_pred.shape =", pers_train_pred.shape)
print("pers_val_pred.shape   =", pers_val_pred.shape)
print("pers_test_pred.shape  =", pers_test_pred.shape)


In [ ]:
def evaluate_output_dict(out, split_name):
    preds_3d = ensure_3d_output(out["preds_real"], N_GEN, HOR)
    trues_3d = ensure_3d_output(out["trues_real"], N_GEN, HOR)
    rows = []
    m = mae_rmse_nmae(trues_3d, preds_3d)
    m.update({"split": split_name, "model": "SpatialElecGNN_global"})
    rows.append(m)
    for j, bus_idx in enumerate(CONFIG["GENERATOR_BUS_IDX"]):
        m = mae_rmse_nmae(trues_3d[:, j, :], preds_3d[:, j, :])
        m.update({"split": split_name, "model": f"SpatialElecGNN_bus_{bus_idx}",
                  "bus_global_idx": bus_idx, "bus_name": buses_df.iloc[bus_idx]["Name"]})
        rows.append(m)
    return pd.DataFrame(rows)

def evaluate_baseline(pred, true, split_name):
    pred_3d = ensure_3d_output(pred, N_GEN, HOR)
    true_3d = ensure_3d_output(true, N_GEN, HOR)
    rows = []
    m = mae_rmse_nmae(true_3d, pred_3d)
    m.update({"split": split_name, "model": "Persistence_global"})
    rows.append(m)
    for j, bus_idx in enumerate(CONFIG["GENERATOR_BUS_IDX"]):
        m = mae_rmse_nmae(true_3d[:, j, :], pred_3d[:, j, :])
        m.update({"split": split_name, "model": f"Persistence_bus_{bus_idx}",
                  "bus_global_idx": bus_idx, "bus_name": buses_df.iloc[bus_idx]["Name"]})
        rows.append(m)
    return pd.DataFrame(rows)

metrics_df = pd.concat([
    evaluate_output_dict(train_out, "train"),
    evaluate_output_dict(val_out,   "val"),
    evaluate_output_dict(test_out,  "test"),
    evaluate_baseline(pers_train_pred, pers_train_true, "train"),
    evaluate_baseline(pers_val_pred,   pers_val_true,   "val"),
    evaluate_baseline(pers_test_pred,  pers_test_true,  "test"),
], ignore_index=True)

display(metrics_df)
print("\nRésumé test (global) :")
display(
    metrics_df[
        (metrics_df["split"] == "test") &
        (metrics_df["model"].str.contains("global"))
    ]
)


In [ ]:
def diagnose_alignment(out, n_checks=5):
    preds_3d = ensure_3d_output(out["preds_real"], N_GEN, HOR)
    trues_3d = ensure_3d_output(out["trues_real"], N_GEN, HOR)
    metas    = np.asarray(out["metas"]).reshape(-1)
    n = min(n_checks, len(metas), len(trues_3d))
    for k in range(n):
        ts0      = int(metas[k])
        expected = p_bus_generator[ts0 : ts0 + HOR].T
        assert np.allclose(expected, trues_3d[k], atol=1e-6), f"Erreur alignement k={k}"
        print(f"k={k} | target_start={ts0} | {ts_p[ts0]} → {ts_p[ts0+HOR-1]} | OK")

diagnose_alignment(test_out, n_checks=CONFIG["NUM_ALIGNMENT_CHECKS"])
print("Diagnostic d'alignement test OK.")


In [ ]:
def per_horizon_metrics(y_true, y_pred):
    y_true_3d = ensure_3d_output(y_true, N_GEN, HOR)
    y_pred_3d = ensure_3d_output(y_pred, N_GEN, HOR)
    rows = []
    for h in range(HOR):
        m = mae_rmse_nmae(y_true_3d[:, :, h], y_pred_3d[:, :, h])
        rows.append({"h": h + 1, **m})
    return pd.DataFrame(rows)

test_horizon_df      = per_horizon_metrics(test_out["trues_real"], test_out["preds_real"])
pers_test_horizon_df = per_horizon_metrics(pers_test_true, pers_test_pred)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(test_horizon_df["h"], test_horizon_df["RMSE"], label="SpatialElecGNN")
ax1.set(xlabel="Horizon (h)", ylabel="RMSE", title="RMSE par horizon - test")
ax1.legend(); ax1.grid(True)

ax2.plot(test_horizon_df["h"], test_horizon_df["MAE"], label="SpatialElecGNN")
ax2.set(xlabel="Horizon (h)", ylabel="MAE", title="MAE par horizon - test")
ax2.legend(); ax2.grid(True)

plt.tight_layout(); plt.show()
display(test_horizon_df.head())


In [ ]:
true_cont = np.full((T, N_GEN), np.nan)
pred_cont = np.full((T, N_GEN), np.nan)

preds_3d = ensure_3d_output(test_out["preds_real"], N_GEN, HOR)
metas    = test_out["metas"]
assert len(metas) == len(preds_3d)

for k, ts0 in enumerate(metas):
    ts0 = int(ts0)
    for j in range(N_GEN):
        true_cont[ts0 : ts0 + HOR, j] = p_bus_generator[ts0 : ts0 + HOR, j]
        pred_cont[ts0 : ts0 + HOR, j] = preds_3d[k, j, :]

n_valid = int(np.sum(np.any(~np.isnan(true_cont), axis=1)))
print(f"Reconstruction continue : {n_valid} heures valides sur {T}")


In [ ]:
N_ZOOM = 72
valid_hours_mask = np.any(~np.isnan(true_cont), axis=1)
valid_idx        = np.where(valid_hours_mask)[0]
start_plot       = valid_idx[0]
end_plot         = min(start_plot + N_ZOOM, T)

print("Zoom 72h de", ts_p[start_plot], "à", ts_p[end_plot - 1])

for j in range(min(4, N_GEN)):
    bus_idx  = CONFIG["GENERATOR_BUS_IDX"][j]
    bus_name = buses_df.iloc[bus_idx]["Name"]
    plt.figure(figsize=(14, 4))
    plt.plot(ts_p[start_plot:end_plot], true_cont[start_plot:end_plot, j],
             label=f"True bus {bus_idx+1} ({bus_name})")
    plt.plot(ts_p[start_plot:end_plot], pred_cont[start_plot:end_plot, j],
             label=f"Pred bus {bus_idx+1} ({bus_name})")
    plt.title(f"Zoom 72h - bus {bus_idx+1} {bus_name}")
    plt.xlabel("Time"); plt.ylabel("p (MW)"); plt.grid(True); plt.legend()
    plt.show()


In [ ]:
true_total_72h = np.nansum(true_cont[start_plot:end_plot, :], axis=1)
pred_total_72h = np.nansum(pred_cont[start_plot:end_plot, :], axis=1)

plt.figure(figsize=(14, 4))
plt.plot(ts_p[start_plot:end_plot], true_total_72h, label="True total 15 bus")
plt.plot(ts_p[start_plot:end_plot], pred_total_72h, label="Pred total 15 bus")
plt.title("Zoom 72h reconstruction continue - total des 15 bus générateurs")
plt.xlabel("Time"); plt.ylabel("Total p (MW)"); plt.grid(True); plt.legend()
plt.show()


In [ ]:
def aggregated_total_metrics(true_arr, pred_arr, n_gen, horizon):
    true_3d = ensure_3d_output(true_arr, n_gen, horizon)
    pred_3d = ensure_3d_output(pred_arr, n_gen, horizon)
    true_total = true_3d.sum(axis=1)
    pred_total = pred_3d.sum(axis=1)
    err_total  = pred_total - true_total
    mae_total  = np.mean(np.abs(err_total))
    rmse_total = np.sqrt(np.mean(err_total ** 2))
    mean_abs   = np.mean(np.abs(true_total))
    return {
        "MAE_total_agg" : mae_total,
        "RMSE_total_agg": rmse_total,
        "STD_error_agg" : np.std(err_total),
        "mean_abs_true" : mean_abs,
        "NMAE_total_agg": mae_total / (mean_abs + 1e-12),
    }, true_total, pred_total, err_total

gnn_metrics,  gnn_true_tot,  gnn_pred_tot,  _ = aggregated_total_metrics(
    test_out["trues_real"], test_out["preds_real"], N_GEN, HOR)
pers_metrics, pers_true_tot, pers_pred_tot, _ = aggregated_total_metrics(
    pers_test_true, pers_test_pred, N_GEN, HOR)

df_total_metrics = pd.DataFrame([
    {"model": "SpatialElecGNN", **gnn_metrics},
    {"model": "Persistence",    **pers_metrics},
])
display(df_total_metrics)


In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.asarray(y_true).reshape(-1) - np.asarray(y_pred).reshape(-1)) ** 2)))

def mae(y_true, y_pred):
    return float(np.mean(np.abs(np.asarray(y_true).reshape(-1) - np.asarray(y_pred).reshape(-1))))

def roughness(x):
    x = np.asarray(x).reshape(-1)
    return float(np.mean(np.abs(np.diff(x)))) if len(x) >= 2 else 0.0

pred_real_3d = ensure_3d_output(val_out["preds_real"], N_GEN, HOR)
true_real_3d = ensure_3d_output(val_out["trues_real"], N_GEN, HOR)
p_pred_total = pred_real_3d.sum(axis=1).reshape(-1)
p_true_total = true_real_3d.sum(axis=1).reshape(-1)

metric_mae      = mae(p_true_total, p_pred_total)
metric_rmse     = rmse(p_true_total, p_pred_total)
rough_true      = roughness(p_true_total)
rough_pred      = roughness(p_pred_total)

metrics_val = {
    "MAE"         : metric_mae,
    "RMSE"        : metric_rmse,
    "MAE/mean"    : metric_mae / (np.mean(np.abs(p_true_total)) + 1e-12),
    "Roughness err": abs(rough_pred - rough_true),
}

print("=== SpatialElecGNN | Global metrics on validation (total des 15 bus générateurs) ===")
for k, v in metrics_val.items():
    print(f"{k:<15}: {v:.6f}")
print(f"{'Roughness true':<15}: {rough_true:.6f}")
print(f"{'Roughness pred':<15}: {rough_pred:.6f}")

names, values = list(metrics_val.keys()), list(metrics_val.values())
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(names, values)
ax.set_title("SpatialElecGNN — Global metrics on validation (p_total agrégé)")
ax.set_ylabel("Value"); ax.grid(True, axis="y", alpha=0.3)
ymax = max(values) if values else 1.0
for b, v in zip(bars, values):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.01 * ymax,
            f"{v:.4f}", ha="center", va="bottom", fontsize=10)
fig.tight_layout()

save_dir  = CONFIG["BASE_DIR"] / "figures"
save_dir.mkdir(parents=True, exist_ok=True)
save_path = save_dir / "global_metrics_val_spatial_elec_gnn.png"
fig.savefig(save_path, dpi=140, bbox_inches="tight")
plt.show()
print("Figure saved to:", save_path.resolve())


In [ ]:
pred_3d = ensure_3d_output(val_out["preds_real"], N_GEN, HOR)
true_3d = ensure_3d_output(val_out["trues_real"], N_GEN, HOR)

std_true_global      = float(np.std(true_3d))
std_pred_global      = float(np.std(pred_3d))
true_total_v         = true_3d.sum(axis=1)
pred_total_v         = pred_3d.sum(axis=1)
std_true_total_global = float(np.std(true_total_v))
std_pred_total_global = float(np.std(pred_total_v))

std_true_h       = np.std(true_3d, axis=(0, 1))
std_pred_h       = np.std(pred_3d, axis=(0, 1))
std_true_total_h = np.std(true_total_v, axis=0)
std_pred_total_h = np.std(pred_total_v, axis=0)
std_true_gen     = np.std(true_3d, axis=(0, 2))
std_pred_gen     = np.std(pred_3d, axis=(0, 2))
std_diff_gen_h   = np.std(pred_3d, axis=0) - np.std(true_3d, axis=0)

print(f"STD globale true : {std_true_global:.4f}  pred : {std_pred_global:.4f}")
print(f"STD totale agrégée true : {std_true_total_global:.4f}  pred : {std_pred_total_global:.4f}")

gen_diff = std_pred_gen - std_true_gen
for label, idxs in [("sous-variés", np.argsort(gen_diff)[:5]),
                     ("sur-variés",  np.argsort(gen_diff)[-5:])]:
    print(f"\nGénérateurs les plus {label}:")
    for idx in idxs:
        print(f"  Gen {idx:02d} | std_true={std_true_gen[idx]:.4f} | std_pred={std_pred_gen[idx]:.4f} | diff={gen_diff[idx]:.4f}")

fig = plt.figure(figsize=(16, 12))
w = 0.35

ax1 = plt.subplot(2, 2, 1)
labels     = ["Global all", "Global total"]
true_vals  = [std_true_global, std_true_total_global]
pred_vals  = [std_pred_global, std_pred_total_global]
x = np.arange(2)
ax1.bar(x - w/2, true_vals, width=w, label="True")
ax1.bar(x + w/2, pred_vals, width=w, label="Pred")
ax1.set_xticks(x); ax1.set_xticklabels(labels)
ax1.set_title("STD globale"); ax1.set_ylabel("STD")
ax1.grid(True, axis="y", alpha=0.3); ax1.legend()

ax2 = plt.subplot(2, 2, 2)
h = np.arange(1, HOR + 1)
ax2.plot(h, std_true_h,       label="True - all outputs")
ax2.plot(h, std_pred_h,       label="Pred - all outputs")
ax2.plot(h, std_true_total_h, linestyle="--", label="True - total agg")
ax2.plot(h, std_pred_total_h, linestyle="--", label="Pred - total agg")
ax2.set_title("STD par horizon"); ax2.set_xlabel("Horizon"); ax2.set_ylabel("STD")
ax2.grid(True, alpha=0.3); ax2.legend()

ax3 = plt.subplot(2, 2, 3)
g = np.arange(N_GEN)
ax3.bar(g - w/2, std_true_gen, width=w, label="True")
ax3.bar(g + w/2, std_pred_gen, width=w, label="Pred")
ax3.set_title("STD par générateur"); ax3.set_xlabel("Generator index"); ax3.set_ylabel("STD")
ax3.grid(True, axis="y", alpha=0.3); ax3.legend()

ax4 = plt.subplot(2, 2, 4)
im = ax4.imshow(std_diff_gen_h, aspect="auto", origin="lower")
ax4.set_title("Heatmap: STD(pred) - STD(true)")
ax4.set_xlabel("Horizon"); ax4.set_ylabel("Generator index")
ax4.set_xticks(np.arange(HOR)); ax4.set_xticklabels(np.arange(1, HOR + 1))
fig.colorbar(im, ax=ax4, fraction=0.046, pad=0.04)

fig.suptitle("STD analysis — SpatialElecGNN (validation)", fontsize=14)
fig.tight_layout(); plt.show()


In [ ]:
ts_                = ts_p
dyn_tensor_        = dyn_tensor
dyn_feature_names_ = dyn_feature_names
p_target_          = p_bus_generator
GEN_IDX_           = CONFIG["GENERATOR_BUS_IDX"]
N_GEN_             = N_GEN
HOR_               = HOR
test_out_          = test_out
pers_test_pred_    = pers_test_pred
pers_test_true_    = pers_test_true
buses_df_          = buses_df

def _to_3d_pred(arr, n_gen, hor):
    return ensure_3d_output(arr, n_gen, hor)

def mae_rmse(y_true, y_pred):
    y_true_f = np.asarray(y_true).reshape(-1)
    y_pred_f = np.asarray(y_pred).reshape(-1)
    return {"MAE": float(np.mean(np.abs(y_true_f - y_pred_f))),
            "RMSE": float(np.sqrt(np.mean((y_true_f - y_pred_f)**2)))}

print("Aliases diagnostics définis (ts_, dyn_tensor_, p_target_, test_out_, etc.)")


In [ ]:
n_feat_show = min(6, dyn_tensor_.shape[-1])
feat_names  = dyn_feature_names_[:n_feat_show]

fig, axes = plt.subplots(2, n_feat_show, figsize=(4 * n_feat_show, 8))
for k in range(n_feat_show):
    vals = dyn_tensor_[..., k].reshape(-1)
    vals = vals[np.isfinite(vals)]
    axes[0, k].hist(vals, bins=60)
    axes[0, k].set_title(f"Hist {feat_names[k]}")
    axes[0, k].grid(True)
    sample = vals[np.linspace(0, len(vals)-1, min(4000, len(vals))).astype(int)]
    axes[1, k].boxplot(sample, vert=True)
    axes[1, k].set_title(f"Box {feat_names[k]}")
    axes[1, k].grid(True)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
target_vals = p_target_.reshape(-1)
target_vals = target_vals[np.isfinite(target_vals)]
axes[0].hist(target_vals, bins=80)
axes[0].set_title("Distribution globale des cibles"); axes[0].grid(True)
axes[1].boxplot(target_vals[np.linspace(0, len(target_vals)-1, min(4000, len(target_vals))).astype(int)], vert=True)
axes[1].set_title("Boxplot global des cibles"); axes[1].grid(True)
plt.tight_layout(); plt.show()

corr_dict = {}
for i in range(n_feat_show):
    corr_dict[f"{feat_names[i]}_mean"] = np.nanmean(dyn_tensor_[:, :, i], axis=1)
for j in range(min(6, p_target_.shape[1])):
    corr_dict[f"target_bus_{GEN_IDX_[j]}"] = p_target_[:, j]

corr_df  = pd.DataFrame(corr_dict)
corr_mat = corr_df.corr(numeric_only=True).values
plt.figure(figsize=(10, 8))
plt.imshow(corr_mat, aspect="auto")
plt.colorbar(label="corr")
plt.xticks(range(len(corr_df.columns)), corr_df.columns, rotation=90)
plt.yticks(range(len(corr_df.columns)), corr_df.columns)
plt.title("Heatmap de corrélations (features agrégées + cibles)")
plt.tight_layout(); plt.show()
display(corr_df.describe().T)


In [ ]:
y_true3 = _to_3d_pred(test_out_["trues_real"], N_GEN_, HOR_)
y_pred3 = _to_3d_pred(test_out_["preds_real"], N_GEN_, HOR_)
b_true3 = _to_3d_pred(pers_test_true_,         N_GEN_, HOR_)
b_pred3 = _to_3d_pred(pers_test_pred_,         N_GEN_, HOR_)

rows = []
for h in range(HOR_):
    m = mae_rmse(y_true3[:, :, h], y_pred3[:, :, h])
    b = mae_rmse(b_true3[:, :, h], b_pred3[:, :, h])
    rows.append({
        "horizon_h"              : h + 1,
        "model_MAE"              : m["MAE"],
        "model_RMSE"             : m["RMSE"],
        "baseline_MAE"           : b["MAE"],
        "baseline_RMSE"          : b["RMSE"],
        "RMSE_gain_vs_baseline_%": 100 * (b["RMSE"] - m["RMSE"]) / (b["RMSE"] + 1e-12),
    })
horizon_df = pd.DataFrame(rows)
display(horizon_df)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(horizon_df["horizon_h"], horizon_df["model_RMSE"],    label="modèle")
axes[0].plot(horizon_df["horizon_h"], horizon_df["baseline_RMSE"], label="baseline")
axes[0].set_title("RMSE par horizon"); axes[0].set_xlabel("Horizon (h)")
axes[0].grid(True); axes[0].legend()

axes[1].plot(horizon_df["horizon_h"], horizon_df["model_MAE"],    label="modèle")
axes[1].plot(horizon_df["horizon_h"], horizon_df["baseline_MAE"], label="baseline")
axes[1].set_title("MAE par horizon"); axes[1].set_xlabel("Horizon (h)")
axes[1].grid(True); axes[1].legend()
plt.tight_layout(); plt.show()

sel = np.linspace(0, len(y_true3)-1, min(3, len(y_true3))).astype(int)
for k in sel:
    plt.figure(figsize=(10, 4))
    plt.plot(np.arange(1, HOR_+1), np.sum(y_true3[k], axis=0), label="true total")
    plt.plot(np.arange(1, HOR_+1), np.sum(y_pred3[k], axis=0), label="pred total")
    plt.plot(np.arange(1, HOR_+1), np.sum(b_pred3[k], axis=0), label="baseline total")
    plt.title(f"Profil 24h | fenêtre test {k}"); plt.xlabel("Horizon (h)")
    plt.grid(True); plt.legend(); plt.show()


In [ ]:
y_true3 = _to_3d_pred(test_out_["trues_real"], N_GEN_, HOR_)
y_pred3 = _to_3d_pred(test_out_["preds_real"], N_GEN_, HOR_)

rows = []
for j in range(N_GEN_):
    yt = y_true3[:, j, :]
    yp = y_pred3[:, j, :]
    m  = mae_rmse(yt, yp)
    rows.append({
        "gen_bus_idx"      : GEN_IDX_[j],
        "MAE"              : m["MAE"],
        "RMSE"             : m["RMSE"],
        "mean_abs_target"  : float(np.mean(np.abs(yt))),
        "NMAE"             : float(m["MAE"] / (np.mean(np.abs(yt)) + 1e-12)),
        "std_true"         : float(np.std(yt)),
        "std_pred"         : float(np.std(yp)),
        "std_ratio_pred_true": float(np.std(yp) / (np.std(yt) + 1e-12)),
    })
node_df = pd.DataFrame(rows).sort_values("RMSE", ascending=False)
display(node_df)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(node_df["gen_bus_idx"].astype(str), node_df["RMSE"])
axes[0].set_title("RMSE par bus générateur"); axes[0].set_xlabel("Bus")
axes[0].tick_params(axis="x", rotation=90); axes[0].grid(True, axis="y")

axes[1].bar(node_df["gen_bus_idx"].astype(str), node_df["std_ratio_pred_true"])
axes[1].axhline(1.0, linestyle="--")
axes[1].set_title("std(pred) / std(true) par bus"); axes[1].set_xlabel("Bus")
axes[1].tick_params(axis="x", rotation=90); axes[1].grid(True, axis="y")
plt.tight_layout(); plt.show()

print("Pires variables cibles (bus) :")
display(node_df.head(10))


In [ ]:
y_true3 = _to_3d_pred(test_out_["trues_real"], N_GEN_, HOR_)
y_pred3 = _to_3d_pred(test_out_["preds_real"], N_GEN_, HOR_)
abs_err = np.abs(y_pred3 - y_true3)

thr          = np.nanpercentile(np.abs(y_true3), 95)
mask_extreme = np.abs(y_true3) >= thr
mask_normal  = np.abs(y_true3) <  thr

extreme_mae = float(np.mean(abs_err[mask_extreme])) if np.any(mask_extreme) else np.nan
normal_mae  = float(np.mean(abs_err[mask_normal]))  if np.any(mask_normal)  else np.nan

display(pd.DataFrame([{
    "threshold_abs_y_95pct"   : thr,
    "MAE_normal_values"       : normal_mae,
    "MAE_extreme_values"      : extreme_mae,
    "extreme_over_normal_ratio": extreme_mae / (normal_mae + 1e-12),
}]))

n_win, n_gen, n_hor = y_true3.shape
flat_idx = np.argsort(abs_err.reshape(-1))[-20:]
coords   = np.array(np.unravel_index(flat_idx, (n_win, n_gen, n_hor))).T
rows = []
for w, j, h in coords[::-1]:
    rows.append({"window_idx": int(w), "bus_idx": int(GEN_IDX_[j]),
                 "horizon_h": int(h+1), "y_true": float(y_true3[w,j,h]),
                 "y_pred": float(y_pred3[w,j,h]), "abs_error": float(abs_err[w,j,h])})
top_err_df = pd.DataFrame(rows)
display(top_err_df)

for _, r in top_err_df.head(3).iterrows():
    w = int(r["window_idx"])
    j = list(GEN_IDX_).index(int(r["bus_idx"]))
    plt.figure(figsize=(10, 4))
    plt.plot(np.arange(1, HOR_+1), y_true3[w, j], label="true")
    plt.plot(np.arange(1, HOR_+1), y_pred3[w, j], label="pred")
    plt.axvline(int(r["horizon_h"]), linestyle="--", label="erreur max")
    plt.title(f"Extrême | fenêtre={w} | bus={GEN_IDX_[j]} | err={r['abs_error']:.3f}")
    plt.xlabel("Horizon (h)"); plt.grid(True); plt.legend(); plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

plt.rcParams.update({
    "figure.figsize": (14, 5), "axes.titlesize": 17, "axes.labelsize": 14,
    "xtick.labelsize": 12, "ytick.labelsize": 12, "legend.fontsize": 12,
    "figure.titlesize": 19, "lines.linewidth": 2.0, "axes.grid": True,
    "grid.alpha": 0.35, "axes.spines.top": False, "axes.spines.right": False,
})

def _get(*names, default=None):
    for n in names:
        if n in globals(): return globals()[n]
    return default

def _to3d(arr, n_gen, hor):
    arr = np.asarray(arr)
    if arr.ndim == 3: return arr
    if arr.ndim == 2: return arr.reshape(arr.shape[0] // n_gen, n_gen, hor)
    raise ValueError(f"Cannot reshape ndim={arr.ndim}")

def _mae(a, b):
    return float(np.mean(np.abs(np.asarray(a).ravel() - np.asarray(b).ravel())))

def _rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a).ravel() - np.asarray(b).ravel())**2)))

N_GEN_  = _get("N_GEN",  default=15)
HOR_    = _get("HOR", "HORIZON_HOURS", default=24)
CONFIG_ = _get("CONFIG", default={})
ts_     = np.asarray(_get("ts_p", "ts", default=np.arange(8784)))

train_out_ = _get("train_out")
val_out_   = _get("val_out")
test_out_  = _get("test_out")

p_target_          = _get("p_bus_generator")
dyn_tensor_        = _get("dyn_tensor")
dyn_feature_names_ = _get("dyn_feature_names")
history_           = _get("history")
GEN_IDX_           = _get("GENERATOR_BUS_IDX", default=list(range(N_GEN_)))
buses_df_          = _get("buses_df")

def bus_name(global_idx):
    if buses_df_ is not None:
        try: return str(buses_df_.iloc[global_idx]["Name"])
        except Exception: pass
    return f"Bus{global_idx+1}"

def autocorr_1d(y, lag=1):
    y = np.asarray(y, dtype=float)
    valid = y[~np.isnan(y)]
    if len(valid) <= lag: return np.nan
    return float(np.corrcoef(valid[:-lag], valid[lag:])[0, 1])

print("=" * 70)
print("GIANT DIAGNOSTIC CELL — GNN Spatial Elec")
print("=" * 70)

# ── Section 1: Learning curves ───────────────────────────────────────────────
print("\n--- SECTION 1: Learning curves ---")
if history_ is not None:
    train_loss = np.asarray(history_["train_loss"], dtype=float)
    val_loss   = np.asarray(history_["val_loss"],   dtype=float)
    lr_hist    = np.asarray(history_.get("lr", []), dtype=float)
    epochs     = np.arange(1, len(train_loss) + 1)
    best_epoch = int(np.argmin(val_loss)) + 1
    best_val   = float(val_loss[best_epoch - 1])
    gap_final  = float(val_loss[-1] - train_loss[-1])

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    axes[0].plot(epochs, train_loss, label="Train loss", color="#2166ac", lw=2)
    axes[0].plot(epochs, val_loss,   label="Val loss",   color="#d73027", lw=2)
    axes[0].axvline(best_epoch, color="green", ls="--", alpha=0.7,
                    label=f"Best epoch={best_epoch} (val={best_val:.4f})")
    axes[0].set_title("Training & Validation Loss (MSE)")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("MSE Loss"); axes[0].legend()
    if len(lr_hist):
        axes[1].plot(epochs[:len(lr_hist)], lr_hist, color="#4d9221", lw=2)
        axes[1].set_title("Learning Rate Schedule")
        axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Learning Rate")
        axes[1].set_yscale("log")
    plt.suptitle("Section 1 — Learning Curves", fontweight="bold")
    plt.tight_layout(); plt.show()
    print(f"  Best epoch (val) : {best_epoch}  →  val_loss = {best_val:.5f}")
    print(f"  Final gap (val-train) : {gap_final:.5f}")
else:
    print("  [SKIP] history_ not found.")

# ── Section 2: Global prediction quality per split ───────────────────────────
print("\n--- SECTION 2: Global prediction quality per split ---")
split_metrics = []
for split_name, out in [("train", train_out_), ("val", val_out_), ("test", test_out_)]:
    if out is None: continue
    y_true = _to3d(out["trues_real"], N_GEN_, HOR_).ravel()
    y_pred = _to3d(out["preds_real"], N_GEN_, HOR_).ravel()
    split_metrics.append({
        "Split": split_name, "MAE": _mae(y_true, y_pred), "RMSE": _rmse(y_true, y_pred),
        "R2": float(r2_score(y_true, y_pred)),
        "std_pred/true": float(np.std(y_pred)) / (float(np.std(y_true)) + 1e-9),
        "Bias": float(np.mean(y_pred - y_true)),
    })
if split_metrics:
    df_spl = pd.DataFrame(split_metrics)
    print(df_spl.to_string(index=False, float_format="{:.4f}".format))
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    splits = df_spl["Split"].str.upper().values
    x = np.arange(len(splits))
    axes[0].bar(x, df_spl["MAE"],  color=["#4575b4","#74add1","#abd9e9"])
    axes[0].set_xticks(x); axes[0].set_xticklabels(splits)
    axes[0].set_title("MAE per Split"); axes[0].set_ylabel("MAE")
    axes[1].bar(x, df_spl["RMSE"], color=["#d73027","#f46d43","#fdae61"])
    axes[1].set_xticks(x); axes[1].set_xticklabels(splits)
    axes[1].set_title("RMSE per Split"); axes[1].set_ylabel("RMSE")
    axes[2].bar(x, df_spl["std_pred/true"], color=["#1a9641","#a6d96a","#ffffbf"])
    axes[2].axhline(1.0, color="black", ls="--", lw=1.5, label="Ideal = 1")
    axes[2].axhline(0.8, color="orange", ls="--", lw=1.5, label="Under-var threshold 0.8")
    axes[2].set_xticks(x); axes[2].set_xticklabels(splits)
    axes[2].set_title("std(pred) / std(true)\n(< 1 = over-smoothing)")
    axes[2].set_ylabel("STD ratio"); axes[2].legend()
    plt.suptitle("Section 2 — Global Prediction Quality per Split", fontweight="bold")
    plt.tight_layout(); plt.show()

# ── Section 3: Error by horizon ──────────────────────────────────────────────
print("\n--- SECTION 3: Error per prediction horizon (test set) ---")
if test_out_ is not None:
    y_true3 = _to3d(test_out_["trues_real"], N_GEN_, HOR_)
    y_pred3 = _to3d(test_out_["preds_real"], N_GEN_, HOR_)
    horizon_rows = [{"h": h+1, "model_MAE": _mae(y_true3[:,:,h], y_pred3[:,:,h]),
                     "model_RMSE": _rmse(y_true3[:,:,h], y_pred3[:,:,h])}
                    for h in range(HOR_)]
    hor_df = pd.DataFrame(horizon_rows)
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    axes[0].plot(hor_df["h"], hor_df["model_MAE"],  color="#2166ac", lw=2.5, marker="o", ms=4)
    axes[0].set_title("MAE per Horizon (Test set)"); axes[0].set_xlabel("Prediction horizon (h)"); axes[0].set_ylabel("MAE")
    axes[1].plot(hor_df["h"], hor_df["model_RMSE"], color="#d73027", lw=2.5, marker="s", ms=4)
    axes[1].set_title("RMSE per Horizon (Test set)"); axes[1].set_xlabel("Prediction horizon (h)"); axes[1].set_ylabel("RMSE")
    plt.suptitle("Section 3 — Horizon-Wise Error (Test Set)", fontweight="bold")
    plt.tight_layout(); plt.show()
    print(f"  RMSE h=1: {hor_df.iloc[0]['model_RMSE']:.4f}  |  RMSE h=24: {hor_df.iloc[-1]['model_RMSE']:.4f}")

# ── Section 4: STD ratio per node ────────────────────────────────────────────
print("\n--- SECTION 4: Smoothing analysis ---")
if test_out_ is not None:
    y_true3 = _to3d(test_out_["trues_real"], N_GEN_, HOR_)
    y_pred3 = _to3d(test_out_["preds_real"], N_GEN_, HOR_)
    node_std_rows = []
    for j in range(N_GEN_):
        std_t  = float(np.std(y_true3[:, j, :]))
        std_p  = float(np.std(y_pred3[:, j, :]))
        gen_idx = GEN_IDX_[j] if j < len(GEN_IDX_) else j
        node_std_rows.append({"bus_j": j, "global_bus_idx": gen_idx,
                               "bus_name": bus_name(gen_idx),
                               "std_true": std_t, "std_pred": std_p,
                               "ratio": std_p / (std_t + 1e-9)})
    node_std_df = pd.DataFrame(node_std_rows).sort_values("ratio")
    pct_under   = float((node_std_df["ratio"] < 0.8).mean()) * 100
    agg_true    = y_true3.sum(axis=1)
    agg_pred    = y_pred3.sum(axis=1)
    agg_std_ratio = float(np.std(agg_pred)) / (float(np.std(agg_true)) + 1e-9)
    print(f"  Mean per-node STD ratio : {node_std_df['ratio'].mean():.3f}")
    print(f"  Aggregated STD ratio    : {agg_std_ratio:.3f}")
    print(f"  % buses with ratio < 0.8: {pct_under:.1f}%")

    fig, axes = plt.subplots(1, 2, figsize=(18, 5))
    colors_bar = ["#d73027" if r < 0.8 else "#4d9221" for r in node_std_df["ratio"]]
    axes[0].barh(range(len(node_std_df)), node_std_df["ratio"], color=colors_bar)
    axes[0].axvline(1.0, color="black", ls="-", lw=1.5, label="Ideal = 1")
    axes[0].axvline(0.8, color="orange", ls="--", lw=1.5, label="Under-var 0.8")
    axes[0].set_yticks(range(len(node_std_df)))
    axes[0].set_yticklabels(node_std_df["bus_name"].values, fontsize=10)
    axes[0].set_title(f"std(pred)/std(true) per Bus (Test) — {pct_under:.0f}% under-varied")
    axes[0].set_xlabel("STD ratio"); axes[0].legend()
    n_samp = min(3000, len(agg_true.ravel()))
    idx_s  = np.random.choice(len(agg_true.ravel()), n_samp, replace=False)
    lim    = max(np.abs(agg_true.ravel()).max(), np.abs(agg_pred.ravel()).max())
    axes[1].scatter(agg_true.ravel()[idx_s], agg_pred.ravel()[idx_s], alpha=0.25, s=8, color="#4575b4")
    axes[1].plot([0, lim], [0, lim], "k--", lw=1.5, label="Perfect")
    axes[1].set_title(f"Aggregated Total — STD ratio = {agg_std_ratio:.3f}")
    axes[1].set_xlabel("True total"); axes[1].set_ylabel("Predicted total"); axes[1].legend()
    plt.suptitle("Section 4 — Variability Smoothing", fontweight="bold")
    plt.tight_layout(); plt.show()

# ── Section 5: Feature-target correlation ────────────────────────────────────
print("\n--- SECTION 5: Dynamic feature – target correlation (lag = horizon) ---")
if dyn_tensor_ is not None and p_target_ is not None:
    T_data     = min(dyn_tensor_.shape[0], p_target_.shape[0])
    F_dyn      = dyn_tensor_.shape[-1]
    feat_names = dyn_feature_names_ if dyn_feature_names_ is not None else [f"dyn_{i}" for i in range(F_dyn)]
    target_total = np.nansum(p_target_[:T_data], axis=1)
    lag = HOR_
    corr_rows = []
    for k in range(F_dyn):
        feat_mean = np.nanmean(dyn_tensor_[:T_data, :, k], axis=1)
        valid = ~(np.isnan(feat_mean) | np.isnan(target_total))
        x, y  = feat_mean[valid], target_total[valid]
        corr_val = float(np.corrcoef(x[:-lag], y[lag:])[0, 1]) if len(x) > lag + 10 else np.nan
        corr_rows.append({"feature": feat_names[k], f"corr_t+{lag}h": corr_val})
    corr_df  = pd.DataFrame(corr_rows).sort_values(f"corr_t+{lag}h", key=abs, ascending=False)
    col_key  = f"corr_t+{lag}h"
    print(corr_df.to_string(index=False))
    fig, ax = plt.subplots(figsize=(10, 5))
    colors_c = ["#2166ac" if abs(v) > 0.3 else "#d73027" for v in corr_df[col_key]]
    ax.barh(corr_df["feature"], corr_df[col_key], color=colors_c)
    ax.axvline(0.0, color="black", lw=1)
    ax.axvline(0.3, color="green", ls="--", alpha=0.7, label="±0.3")
    ax.axvline(-0.3, color="green", ls="--", alpha=0.7)
    ax.set_title(f"Feature correlation with target t+{lag}h"); ax.legend()
    plt.tight_layout(); plt.show()

# ── Section 6: Autocorrelation & CV ──────────────────────────────────────────
print("\n--- SECTION 6: Intrinsic target difficulty ---")
if p_target_ is not None:
    ac_rows = []
    for j in range(p_target_.shape[1]):
        y = p_target_[:, j]
        mean_abs = float(np.nanmean(np.abs(y)))
        std_y    = float(np.nanstd(y))
        gen_idx  = GEN_IDX_[j] if j < len(GEN_IDX_) else j
        ac_rows.append({
            "bus_name": bus_name(gen_idx), "global_idx": gen_idx,
            "mean_abs": mean_abs, "std": std_y,
            "cv": std_y / (mean_abs + 1e-9),
            "autocorr_1h": autocorr_1d(y, lag=1),
            "autocorr_24h": autocorr_1d(y, lag=24),
        })
    ac_df = pd.DataFrame(ac_rows).sort_values("autocorr_24h")
    fig, axes = plt.subplots(1, 3, figsize=(22, 5))
    axes[0].barh(range(len(ac_df)), ac_df["autocorr_1h"],  color="#4575b4")
    axes[0].set_yticks(range(len(ac_df))); axes[0].set_yticklabels(ac_df["bus_name"].values, fontsize=10)
    axes[0].set_title("Autocorrelation lag 1h"); axes[0].set_xlabel("Autocorr lag=1h")
    axes[0].axvline(0.5, color="orange", ls="--", lw=1.5, label="0.5"); axes[0].legend()
    axes[1].barh(range(len(ac_df)), ac_df["autocorr_24h"], color="#4d9221")
    axes[1].set_yticks(range(len(ac_df))); axes[1].set_yticklabels(ac_df["bus_name"].values, fontsize=10)
    axes[1].set_title("Autocorrelation lag 24h"); axes[1].set_xlabel("Autocorr lag=24h")
    axes[1].axvline(0.4, color="orange", ls="--", lw=1.5, label="0.4"); axes[1].legend()
    cv_sorted = ac_df.sort_values("cv", ascending=False)
    clrs_cv   = ["#d73027" if cv > 1.5 else "#4d9221" for cv in cv_sorted["cv"]]
    axes[2].barh(range(len(cv_sorted)), cv_sorted["cv"], color=clrs_cv)
    axes[2].set_yticks(range(len(cv_sorted))); axes[2].set_yticklabels(cv_sorted["bus_name"].values, fontsize=10)
    axes[2].set_title("CV = std / |mean|  (Red > 1.5)"); axes[2].set_xlabel("CV")
    axes[2].axvline(1.5, color="orange", ls="--", lw=1.5, label="CV=1.5"); axes[2].legend()
    plt.suptitle("Section 6 — Intrinsic Predictability", fontweight="bold")
    plt.tight_layout(); plt.show()

# ── Section 7: Per-node errors ────────────────────────────────────────────────
print("\n--- SECTION 7: Per-node errors (test set) ---")
if test_out_ is not None:
    y_true3 = _to3d(test_out_["trues_real"], N_GEN_, HOR_)
    y_pred3 = _to3d(test_out_["preds_real"], N_GEN_, HOR_)
    node_rows = []
    for j in range(N_GEN_):
        yt = y_true3[:, j, :]
        yp = y_pred3[:, j, :]
        mae_j  = _mae(yt, yp)
        rmse_j = _rmse(yt, yp)
        std_t  = float(np.std(yt))
        std_p  = float(np.std(yp))
        mean_t = float(np.mean(np.abs(yt)))
        gen_idx = GEN_IDX_[j] if j < len(GEN_IDX_) else j
        node_rows.append({"bus_j": j, "global_idx": gen_idx, "bus_name": bus_name(gen_idx),
                           "MAE": mae_j, "RMSE": rmse_j, "NMAE": mae_j / (mean_t + 1e-9),
                           "std_true": std_t, "std_pred": std_p, "std_ratio": std_p / (std_t + 1e-9)})
    node_df    = pd.DataFrame(node_rows).sort_values("RMSE", ascending=False)
    total_rmse = node_df["RMSE"].sum()
    node_df["RMSE_share_%"] = 100 * node_df["RMSE"] / (total_rmse + 1e-9)
    top3_share = float(node_df.head(3)["RMSE_share_%"].sum())
    print(f"  Top-3 worst buses = {top3_share:.1f}% of total RMSE sum")
    fig, axes = plt.subplots(1, 2, figsize=(18, 5))
    node_sort_rmse = node_df.sort_values("RMSE", ascending=True)
    mean_rmse      = total_rmse / N_GEN_
    colors_rmse    = ["#d73027" if r > mean_rmse else "#4575b4" for r in node_sort_rmse["RMSE"]]
    axes[0].barh(range(len(node_sort_rmse)), node_sort_rmse["RMSE"], color=colors_rmse)
    axes[0].set_yticks(range(len(node_sort_rmse))); axes[0].set_yticklabels(node_sort_rmse["bus_name"].values, fontsize=10)
    axes[0].set_title(f"RMSE per Bus (Test) — Top 3 = {top3_share:.1f}%")
    axes[0].axvline(mean_rmse, color="orange", ls="--", lw=1.5, label=f"Mean={mean_rmse:.2f}"); axes[0].legend()
    node_sort_ratio = node_df.sort_values("std_ratio", ascending=True)
    clrs_r = ["#d73027" if r < 0.8 else "#4d9221" for r in node_sort_ratio["std_ratio"]]
    axes[1].barh(range(len(node_sort_ratio)), node_sort_ratio["std_ratio"], color=clrs_r)
    axes[1].axvline(1.0, color="black", ls="-", lw=1.5, label="Ideal = 1")
    axes[1].axvline(0.8, color="orange", ls="--", lw=1.5, label="Under-var 0.8")
    axes[1].set_yticks(range(len(node_sort_ratio))); axes[1].set_yticklabels(node_sort_ratio["bus_name"].values, fontsize=10)
    axes[1].set_title("STD Ratio per Bus (Test)"); axes[1].legend()
    plt.suptitle("Section 7 — Per-Node Error Analysis (Test Set)", fontweight="bold")
    plt.tight_layout(); plt.show()

# ── Section 8: ON/OFF failure analysis ───────────────────────────────────────
print("\n--- SECTION 8: Systematic ON/OFF failure analysis ---")
if test_out_ is not None:
    y_true3 = _to3d(test_out_["trues_real"], N_GEN_, HOR_)
    y_pred3 = _to3d(test_out_["preds_real"], N_GEN_, HOR_)
    abs_err = np.abs(y_pred3 - y_true3)

    assert "CONFIG" in globals() and "GENERATOR_BUS_IDX" in CONFIG
    gen_idx_local = list(CONFIG["GENERATOR_BUS_IDX"])
    gen_bus_names = buses_df.iloc[gen_idx_local]["Name"].tolist()
    assert len(gen_idx_local) == y_true3.shape[1]

    near_zero_thresh = 0.5
    nonzero_pred_thr = 5.0
    mask_off_fail = (np.abs(y_true3) < near_zero_thresh) & (np.abs(y_pred3) > nonzero_pred_thr)
    n_off_fail    = int(mask_off_fail.sum())
    mae_off_fail  = float(np.mean(abs_err[mask_off_fail])) if n_off_fail > 0 else np.nan

    print(f"  Failures : {n_off_fail}  |  MAE on failures : {mae_off_fail:.4f}")

    fail_per_bus = mask_off_fail.sum(axis=(0, 2))
    fail_df = pd.DataFrame({
        "bus_j": np.arange(len(gen_idx_local)),
        "global_bus_idx": gen_idx_local,
        "bus_name": gen_bus_names,
        "n_failures": fail_per_bus,
    }).sort_values("n_failures", ascending=False).reset_index(drop=True)
    display(fail_df)

    worst_bus_j      = int(fail_df.iloc[0]["bus_j"])
    worst_bus_nm     = str(fail_df.iloc[0]["bus_name"])
    fail_per_win     = mask_off_fail[:, worst_bus_j, :].sum(axis=1)

    fig, axes = plt.subplots(1, 3, figsize=(22, 5))
    clrs_fail = ["#d73027" if n > 0 else "#4d9221" for n in fail_df["n_failures"]]
    axes[0].barh(range(len(fail_df)), fail_df["n_failures"], color=clrs_fail)
    axes[0].set_yticks(range(len(fail_df))); axes[0].set_yticklabels(fail_df["bus_name"].values, fontsize=10)
    axes[0].invert_yaxis()
    axes[0].set_title(f"ON/OFF Failure per Generator\ntrue < {near_zero_thresh} & pred > {nonzero_pred_thr}")
    axes[0].set_xlabel("Number of failures"); axes[0].grid(True, axis="x", alpha=0.3)

    yt_w = y_true3[:, worst_bus_j, :].ravel()
    yp_w = y_pred3[:, worst_bus_j, :].ravel()
    clrs_sc = np.where((np.abs(yt_w) < near_zero_thresh) & (np.abs(yp_w) > nonzero_pred_thr), "#d73027", "#4575b4")
    lim_w   = max(np.abs(yt_w).max(), np.abs(yp_w).max())
    axes[1].scatter(yt_w, yp_w, c=clrs_sc, alpha=0.3, s=10)
    axes[1].plot([0, lim_w], [0, lim_w], "k--", lw=1.5, label="Perfect")
    axes[1].set_title(f"True vs Pred — {worst_bus_nm}")
    axes[1].set_xlabel("True production"); axes[1].set_ylabel("Predicted production"); axes[1].legend()

    if fail_per_win.max() > 0:
        worst_win = int(np.argmax(fail_per_win))
        axes[2].plot(range(1, HOR_+1), y_true3[worst_win, worst_bus_j, :], "o-", color="blue", lw=2, label="True")
        axes[2].plot(range(1, HOR_+1), y_pred3[worst_win, worst_bus_j, :], "s--", color="red",  lw=2, label="Predicted")
        axes[2].set_title(f"Worst Failure Window — {worst_bus_nm}")
        axes[2].set_xlabel("Horizon h"); axes[2].set_ylabel("Production"); axes[2].legend()
    else:
        axes[2].text(0.5, 0.5, "No ON/OFF failure detected", ha="center", va="center",
                     transform=axes[2].transAxes, fontsize=13)
    plt.suptitle("Section 8 — Systematic ON/OFF Failures", fontweight="bold")
    plt.tight_layout(); plt.show()

# ── Section 9: Residual analysis ─────────────────────────────────────────────
print("\n--- SECTION 9: Global residual analysis (test set) ---")
if test_out_ is not None:
    y_true_f = _to3d(test_out_["trues_real"], N_GEN_, HOR_).ravel()
    y_pred_f = _to3d(test_out_["preds_real"], N_GEN_, HOR_).ravel()
    resid    = y_pred_f - y_true_f
    bias      = float(np.mean(resid))
    std_err   = float(np.std(resid))
    r2_val    = float(r2_score(y_true_f, y_pred_f))
    std_ratio = float(np.std(y_pred_f)) / (float(np.std(y_true_f)) + 1e-9)
    print(f"  R²={r2_val:.4f}  Bias={bias:.4f}  std(resid)={std_err:.4f}  std ratio={std_ratio:.4f}")

    n_samp = min(6000, len(y_true_f))
    idx_s  = np.random.choice(len(y_true_f), n_samp, replace=False)
    lim_all = max(np.abs(y_true_f).max(), np.abs(y_pred_f).max())
    fig, axes = plt.subplots(1, 3, figsize=(22, 5))
    axes[0].scatter(y_true_f[idx_s], y_pred_f[idx_s], alpha=0.15, s=6, color="#4575b4")
    axes[0].plot([0, lim_all], [0, lim_all], "k--", lw=1.5, label="Perfect")
    axes[0].set_title(f"True vs Pred  R²={r2_val:.3f}  std_ratio={std_ratio:.3f}")
    axes[0].set_xlabel("True"); axes[0].set_ylabel("Pred"); axes[0].legend()
    axes[1].hist(resid, bins=80, color="#4575b4", edgecolor="none", alpha=0.8)
    axes[1].axvline(bias, color="red", ls="--", lw=2, label=f"Bias={bias:.3f}")
    axes[1].axvline(0, color="black", ls="-", lw=1)
    axes[1].set_title(f"Residual Distribution  std={std_err:.3f}")
    axes[1].set_xlabel("Residual"); axes[1].set_ylabel("Count"); axes[1].legend()
    axes[2].scatter(y_true_f[idx_s], resid[idx_s], alpha=0.15, s=6, color="#d73027")
    axes[2].axhline(0, color="black", lw=1)
    axes[2].axhline(bias, color="red", ls="--", lw=2, label=f"Mean bias={bias:.3f}")
    axes[2].set_title("Residual vs True Value")
    axes[2].set_xlabel("True"); axes[2].set_ylabel("Residual"); axes[2].legend()
    plt.suptitle("Section 9 — Global Residual Analysis (Test Set)", fontweight="bold")
    plt.tight_layout(); plt.show()

# ── Section 10: Consolidated summary ─────────────────────────────────────────
print("\n" + "=" * 70)
print("CONSOLIDATED DIAGNOSTIC SUMMARY")
print("=" * 70)
diag_lines = []
if history_ is not None:
    diag_lines.append(f"[MODEL]  Val loss stagnates at epoch ~{best_epoch} (gap={gap_final:.4f}) → overfitting risk.")
if 'agg_std_ratio' in dir():
    diag_lines.append(f"[MODEL]  Aggregated STD ratio = {agg_std_ratio:.3f} → {100*(1-agg_std_ratio):.0f}% variability lost.")
if 'pct_under' in dir():
    diag_lines.append(f"[MODEL]  {pct_under:.0f}% of buses have std ratio < 0.8 → generalised over-smoothing.")
if 'hor_df' in dir():
    diag_lines.append(f"[MODEL]  RMSE h=1→h=24: {hor_df.iloc[0]['model_RMSE']:.3f} → {hor_df.iloc[-1]['model_RMSE']:.3f}  (+{100*(hor_df.iloc[-1]['model_RMSE']-hor_df.iloc[0]['model_RMSE'])/(hor_df.iloc[0]['model_RMSE']+1e-9):.1f}%).")
if 'top3_share' in dir():
    diag_lines.append(f"[MODEL]  Top-3 worst buses = {top3_share:.1f}% of total node RMSE.")
if 'n_off_fail' in dir() and n_off_fail > 0:
    diag_lines.append(f"[DATA]   {n_off_fail} ON/OFF failures detected.")
for i, line in enumerate(diag_lines, 1):
    print(f"  {i:2d}. {line}")
print("=" * 70)


In [ ]:
class NoMessagePassingElec(nn.Module):
    def __init__(self, in_dim, hidden_dim, horizon,
                 encoder_hidden_dims=(256, 128), decoder_hidden_dims=(128, 64),
                 dropout=0.0, use_batchnorm=True,
                 use_day_embedding=False, day_emb_dim=3):
        super().__init__()
        self.use_day_embedding = use_day_embedding
        if use_day_embedding:
            self.day_embedding = nn.Embedding(7, day_emb_dim)
            enc_in_dim = in_dim + day_emb_dim
        else:
            enc_in_dim = in_dim
        enc_dims = [enc_in_dim] + list(encoder_hidden_dims) + [hidden_dim]
        self.encoder = MLP(enc_dims, dropout=dropout, use_batchnorm=use_batchnorm, activate_last=False)
        dec_dims = [hidden_dim] + list(decoder_hidden_dims) + [horizon]
        self.decoder = MLP(dec_dims, dropout=dropout, use_batchnorm=use_batchnorm, activate_last=False)

    def forward(self, x, edge_index, edge_attr, generator_mask, dow=None, batch_idx=None):
        if self.use_day_embedding and dow is not None and batch_idx is not None:
            dow_emb = self.day_embedding(dow)
            x = torch.cat([x, dow_emb[batch_idx]], dim=1)
        h     = self.encoder(x)
        h_gen = h[generator_mask]
        return self.decoder(h_gen)


ablation_model = NoMessagePassingElec(
    in_dim              = expected_x_dim,
    hidden_dim          = CONFIG["HIDDEN_DIM"],
    horizon             = HOR,
    encoder_hidden_dims = CONFIG["ENCODER_HIDDEN_DIMS"],
    decoder_hidden_dims = CONFIG["DECODER_HIDDEN_DIMS"],
    dropout             = CONFIG["DROPOUT"],
    use_batchnorm       = CONFIG["USE_BATCHNORM"],
    use_day_embedding   = use_day_embedding,
    day_emb_dim         = day_emb_dim,
).to(DEVICE)

ablation_optimizer    = torch.optim.Adam(ablation_model.parameters(), lr=CONFIG["LR"], weight_decay=CONFIG["WEIGHT_DECAY"])
ablation_scheduler    = torch.optim.lr_scheduler.ReduceLROnPlateau(
    ablation_optimizer, mode="min", factor=CONFIG["SCHEDULER_FACTOR"],
    patience=CONFIG["SCHEDULER_PATIENCE"], min_lr=CONFIG["SCHEDULER_MIN_LR"])
ablation_early_stopper = EarlyStopping(patience=CONFIG["EARLY_STOPPING_PATIENCE"],
                                       min_delta=CONFIG["EARLY_STOPPING_MIN_DELTA"])
ablation_history = {"train_loss": [], "val_loss": [], "lr": []}

print("=== Modèle sans message passing ===")
print(ablation_model)

for epoch in range(1, CONFIG["EPOCHS"] + 1):
    train_loss = run_one_epoch(ablation_model, train_loader, optimizer=ablation_optimizer)
    val_loss   = run_one_epoch(ablation_model, val_loader, optimizer=None)
    ablation_scheduler.step(val_loss)
    current_lr = ablation_optimizer.param_groups[0]["lr"]
    ablation_history["train_loss"].append(train_loss)
    ablation_history["val_loss"].append(val_loss)
    ablation_history["lr"].append(current_lr)
    print(f"[NoMP] Epoch {epoch:03d} | train={train_loss:.6f} | val={val_loss:.6f} | lr={current_lr:.2e}")
    if ablation_early_stopper.step(val_loss, ablation_model):
        print(f"[NoMP] Early stopping à l'epoch {epoch}.")
        break

assert ablation_early_stopper.best_state is not None
ablation_model.load_state_dict(ablation_early_stopper.best_state)
print("\n[NoMP] Meilleur val_loss :", ablation_early_stopper.best_loss)

ablation_train_out = collect_predictions(ablation_model, train_loader)
ablation_val_out   = collect_predictions(ablation_model, val_loader)
ablation_test_out  = collect_predictions(ablation_model, test_loader)


def evaluate_output_dict_custom(out, split_name, model_prefix):
    preds_3d = ensure_3d_output(out["preds_real"], N_GEN, HOR)
    trues_3d = ensure_3d_output(out["trues_real"], N_GEN, HOR)
    rows = []
    m = mae_rmse_nmae(trues_3d, preds_3d)
    m.update({"split": split_name, "model": f"{model_prefix}_global"})
    rows.append(m)
    for j, bus_idx in enumerate(CONFIG["GENERATOR_BUS_IDX"]):
        m = mae_rmse_nmae(trues_3d[:, j, :], preds_3d[:, j, :])
        m.update({"split": split_name, "model": f"{model_prefix}_bus_{bus_idx}",
                  "bus_global_idx": bus_idx, "bus_name": buses_df.iloc[bus_idx]["Name"]})
        rows.append(m)
    return pd.DataFrame(rows)

ablation_metrics_df = pd.concat([
    evaluate_output_dict_custom(ablation_train_out, "train", "NoMP"),
    evaluate_output_dict_custom(ablation_val_out,   "val",   "NoMP"),
    evaluate_output_dict_custom(ablation_test_out,  "test",  "NoMP"),
], ignore_index=True)

display(ablation_metrics_df)

comparison_rows = []
for model_label, df, model_col in [
    ("NoMessagePassing", ablation_metrics_df, "NoMP_global"),
    ("SpatialElecGNN",   metrics_df,          "SpatialElecGNN_global"),
    ("Persistence",      metrics_df,          "Persistence_global"),
]:
    tmp = df[(df["split"] == "test") & (df["model"] == model_col)]
    if len(tmp) == 1:
        comparison_rows.append({"model": model_label, **{k: tmp.iloc[0][k] for k in ["MAE","RMSE","NMAE"]}})

comparison_df = pd.DataFrame(comparison_rows).sort_values("RMSE").reset_index(drop=True)
print("\n=== Comparaison finale test (global) ===")
display(comparison_df)

if len(comparison_df) >= 2:
    gnn_rmse  = comparison_df.loc[comparison_df["model"] == "SpatialElecGNN",   "RMSE"]
    nomp_rmse = comparison_df.loc[comparison_df["model"] == "NoMessagePassing", "RMSE"]
    if len(gnn_rmse) and len(nomp_rmse):
        rel_gap = 100 * (nomp_rmse.iloc[0] - gnn_rmse.iloc[0]) / max(abs(gnn_rmse.iloc[0]), 1e-12)
        print(f"Écart relatif RMSE NoMP vs GNN = {rel_gap:.2f}%")
        if abs(rel_gap) < 2:
            print("Conclusion : le message passing apporte très peu de gain mesurable.")
        elif rel_gap > 0:
            print("Conclusion : le GNN complet est meilleur ; le message passing apporte de l'information.")
        else:
            print("Conclusion : le modèle sans message passing fait mieux ; message passing inutile ici.")


In [ ]:
def aggregate_totals_per_hour(arr, n_gen, horizon):
    return ensure_3d_output(arr, n_gen, horizon).sum(axis=1)

gnn_test_pred_tot  = aggregate_totals_per_hour(test_out["preds_real"],           N_GEN, HOR)
gnn_test_true_tot  = aggregate_totals_per_hour(test_out["trues_real"],           N_GEN, HOR)
nomp_test_pred_tot = aggregate_totals_per_hour(ablation_test_out["preds_real"],   N_GEN, HOR)
nomp_test_true_tot = aggregate_totals_per_hour(ablation_test_out["trues_real"],   N_GEN, HOR)
pers_test_pred_tot = aggregate_totals_per_hour(pers_test_pred,                    N_GEN, HOR)
pers_test_true_tot = aggregate_totals_per_hour(pers_test_true,                    N_GEN, HOR)

rows = []
for label, true_tot, pred_tot in [
    ("SpatialElecGNN_aggregated_total",  gnn_test_true_tot,  gnn_test_pred_tot),
    ("Persistence_aggregated_total",     pers_test_true_tot, pers_test_pred_tot),
    ("NoMessagePassing_aggregated_total",nomp_test_true_tot, nomp_test_pred_tot),
]:
    m = mae_rmse_nmae(true_tot, pred_tot)
    m["model"] = label
    rows.append(m)

agg_total_df = pd.DataFrame(rows)[["model", "MAE", "RMSE", "NMAE"]].sort_values("RMSE").reset_index(drop=True)
print("=== Comparaison TEST sur les TOTAUX AGRÉGÉS ===")
display(agg_total_df)

gnn_rmse  = agg_total_df.loc[agg_total_df["model"] == "SpatialElecGNN_aggregated_total",   "RMSE"]
nomp_rmse = agg_total_df.loc[agg_total_df["model"] == "NoMessagePassing_aggregated_total", "RMSE"]
if len(gnn_rmse) and len(nomp_rmse):
    rel_gap = 100 * (nomp_rmse.iloc[0] - gnn_rmse.iloc[0]) / max(abs(gnn_rmse.iloc[0]), 1e-12)
    print(f"Écart relatif RMSE NoMP vs GNN (totaux agrégés) = {rel_gap:.2f}%")


In [ ]:
def roughness(x):
    x = np.asarray(x).reshape(-1)
    return float(np.mean(np.abs(np.diff(x)))) if len(x) >= 2 else 0.0

rows = []
for label, true_tot, pred_tot in [
    ("SpatialElecGNN",  gnn_test_true_tot,  gnn_test_pred_tot),
    ("Persistence",     pers_test_true_tot, pers_test_pred_tot),
    ("NoMessagePassing",nomp_test_true_tot, nomp_test_pred_tot),
]:
    rows.append({
        "model"      : label,
        "rough_true" : roughness(true_tot),
        "rough_pred" : roughness(pred_tot),
        "rough_error": roughness(pred_tot - true_tot),
    })

roughness_df = pd.DataFrame(rows)
roughness_df["pred_minus_true"] = roughness_df["rough_pred"] - roughness_df["rough_true"]
roughness_df["pred_over_true"]  = roughness_df["rough_pred"] / np.maximum(roughness_df["rough_true"], 1e-12)
print("=== Roughness sur les TOTAUX AGRÉGÉS (TEST) ===")
display(roughness_df)

print("\n=== Lecture rapide ===")
for _, r in roughness_df.iterrows():
    print(f"{r['model']}:")
    print(f"  rough_true={r['rough_true']:.4f}  rough_pred={r['rough_pred']:.4f}  pred/true={r['pred_over_true']:.3f}")
